<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/DEEPSEEK_V4DOT1_DEMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import time
from openai import OpenAI
from openai import APIConnectionError, AuthenticationError, RateLimitError, APIStatusError

# --- Configuration (Replace with your actual key/model) ---
# NOTE: The API key should be securely stored in Google Colab's Userdata or environment variables.
# For this test, we assume 'DEEPSEEK_API_KEY' is set in the environment.
from google.colab import userdata
DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_MODEL = "deepseek-flash"  # DeepSeek V4.1 Flash
TEST_PROMPT = "Diagnose the connection to the DeepSeek API. Respond with only the word 'OK'."
MAX_RETRIES = 3  # For transient connection errors


def check_deepseek_api_access():
    """Attempts to connect to the DeepSeek API (V4.1 Flash) and reports all error types."""

    print(f"--- DeepSeek API Diagnostic Check ({DEEPSEEK_MODEL}) ---")

    # 1. Check for missing API Key
    if not DEEPSEEK_API_KEY:
        print("❌ CRITICAL FAILURE: API Key not loaded.")
        print("ACTION: Ensure 'DEEPSEEK_API_KEY' is correctly set in your environment.")
        return

    client = OpenAI(
        api_key=DEEPSEEK_API_KEY,
        base_url=DEEPSEEK_BASE_URL,
        timeout=60.0,  # Increased timeout for V4.1 Flash
    )

    # 2. Attempt API Call with Retries for Connection Errors
    for attempt in range(MAX_RETRIES):
        try:
            print(f"[{time.strftime('%H:%M:%S')}] Attempting API call (Retry {attempt + 1})...")

            # Make a minimal chat completion request
            response = client.chat.completions.create(
                model=DEEPSEEK_MODEL,
                messages=[{"role": "user", "content": TEST_PROMPT}],
                stream=False,
                reasoning_effort="low",  # Optional: reduces thinking overhead
            )

            # 3. Analyze Successful Response
            message = response.choices[0].message
            response_text = (message.content or "").strip()
            reasoning = getattr(message, "reasoning_content", None)

            print("\n✅ DIAGNOSIS SUCCESSFUL:")
            print(f"  - Model: {DEEPSEEK_MODEL}")
            print(f"  - Status: 200 OK (Connection, Authentication, and Service Operational)")
            print(f"  - Response: '{response_text}'")
            if reasoning:
                print(f"  - Reasoning content present ({len(reasoning)} chars)")
            print(f"  - Tokens Used: {response.usage.total_tokens}")
            return

        except AuthenticationError:
            print("\n❌ AUTHENTICATION FAILURE (Error 401):")
            print("ACTION: Your API key is incorrect, expired, or hasn't been topped up.")
            print("CHECK: DeepSeek Platform > API Keys/Usage to verify status and balance.")
            return

        except RateLimitError:
            print("\n⚠️ RATE LIMIT FAILURE (Error 429):")
            print("ACTION: You are making too many requests.")
            print("CHECK: Your plan's rate limit (RPM/TPM).")
            return

        except APIStatusError as e:
            if e.status_code == 402:
                print("\n⚠️ INSUFFICIENT BALANCE (Error 402):")
                print("ACTION: Your account balance is too low.")
                print("CHECK: DeepSeek Platform > Usage to top up.")
            elif e.status_code == 429:
                print("\n⚠️ RATE LIMITED (Error 429):")
                print("ACTION: Slow down your requests.")
            else:
                print(f"\n❌ API ERROR {e.status_code}: {e}")
            return

        except APIConnectionError as e:
            if attempt < MAX_RETRIES - 1:
                wait_time = 2 ** attempt
                print(f"❌ CONNECTION FAILURE (APIConnectionError): {e}. Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print("\n❌ CRITICAL FAILURE (APIConnectionError):")
                print(f"   Last error: {e}")
                print("ACTION: Connection failed after multiple retries. The DeepSeek server is likely down or unreachable.")
                print("CHECK: DeepSeek Status Page (https://status.deepseek.com/) and your network connection.")
                return

        except Exception as e:
            print(f"\n❌ UNHANDLED ERROR: {e.__class__.__name__}: {e}")
            print("ACTION: Investigate the error message for specific network or API configuration issues.")
            return


if __name__ == "__main__":
    check_deepseek_api_access()

--- DeepSeek API Diagnostic Check (deepseek-flash) ---
[08:26:53] Attempting API call (Retry 1)...

✅ DIAGNOSIS SUCCESSFUL:
  - Model: deepseek-flash
  - Status: 200 OK (Connection, Authentication, and Service Operational)
  - Response: 'OK'
  - Reasoning content present (164 chars)
  - Tokens Used: 93


## CASE1

In [2]:
import os
import time
import json
from openai import OpenAI
from openai import APIConnectionError, AuthenticationError, RateLimitError, APIStatusError

# --- Configuration (Reference contract from diagnostic script) ---
from google.colab import userdata
DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_MODEL = "deepseek-flash"  # DeepSeek V4.1 Flash
MAX_RETRIES = 3


# ============================================================
# 1. AOCC SIMULATED DATA SOURCES
# ============================================================

FLEET = [
    {"tail": "N101AA", "type": "A320", "location": "JFK", "status": "available"},
    {"tail": "N102AA", "type": "B737", "location": "LAX", "status": "available"},
    {"tail": "N201AA", "type": "A350", "location": "JFK", "status": "maintenance"},
    {"tail": "N202AA", "type": "B787", "location": "ORD", "status": "available"},
]

CREW = [
    {"id": "CPT001", "role": "Captain", "base": "JFK", "duty_hours_remaining": 8.5},
    {"id": "FO001", "role": "First Officer", "base": "JFK", "duty_hours_remaining": 9.0},
    {"id": "CPT002", "role": "Captain", "base": "LAX", "duty_hours_remaining": 3.0},
    {"id": "FA001", "role": "Flight Attendant", "base": "JFK", "duty_hours_remaining": 10.0},
]

FLIGHTS = [
    {"flight": "AA100", "route": "JFK-LAX", "tail": "N101AA", "status": "scheduled", "departure": "2026-09-18T14:00"},
    {"flight": "AA200", "route": "JFK-ORD", "tail": "N202AA", "status": "delayed", "departure": "2026-09-18T15:30"},
    {"flight": "AA300", "route": "LAX-JFK", "tail": "N102AA", "status": "scheduled", "departure": "2026-09-18T16:00"},
]

WEATHER = {
    "JFK": {"condition": "Thunderstorms", "severity": "high", "visibility": "2 miles"},
    "LAX": {"condition": "Clear", "severity": "low", "visibility": "10 miles"},
    "ORD": {"condition": "Light Rain", "severity": "medium", "visibility": "5 miles"},
}


# ============================================================
# 2. TOOL DEFINITIONS
# ============================================================

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_flight_status",
            "description": "Get the current status of a specific flight including delay, aircraft, and route.",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {"type": "string", "description": "Flight number, e.g., AA100"},
                },
                "required": ["flight_number"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_weather",
            "description": "Check current weather conditions at an airport.",
            "parameters": {
                "type": "object",
                "properties": {
                    "airport_code": {"type": "string", "description": "IATA airport code, e.g., JFK"},
                },
                "required": ["airport_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "find_available_aircraft",
            "description": "Find available aircraft at a specific airport.",
            "parameters": {
                "type": "object",
                "properties": {
                    "airport_code": {"type": "string", "description": "Airport to search, e.g., JFK"},
                },
                "required": ["airport_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_crew_availability",
            "description": "Check crew availability and duty time remaining at a base airport.",
            "parameters": {
                "type": "object",
                "properties": {
                    "base": {"type": "string", "description": "Crew base airport, e.g., JFK"},
                },
                "required": ["base"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "propose_recovery_options",
            "description": "Generate ranked recovery options for a disrupted flight.",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {"type": "string", "description": "Disrupted flight number"},
                    "issue": {"type": "string", "description": "Description of the disruption"},
                },
                "required": ["flight_number", "issue"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "estimate_delay_cost",
            "description": "Estimate the operational cost of a delay.",
            "parameters": {
                "type": "object",
                "properties": {
                    "delay_minutes": {"type": "integer", "description": "Delay duration in minutes"},
                    "aircraft_type": {"type": "string", "description": "Aircraft type, e.g., A320"},
                },
                "required": ["delay_minutes"],
            },
        },
    },
]


# ============================================================
# 3. TOOL IMPLEMENTATIONS
# ============================================================

def get_flight_status(flight_number: str) -> str:
    for f in FLIGHTS:
        if f["flight"] == flight_number:
            return json.dumps(f)
    return json.dumps({"error": f"Flight {flight_number} not found"})


def check_weather(airport_code: str) -> str:
    weather = WEATHER.get(airport_code.upper())
    if weather:
        return json.dumps({"airport": airport_code, **weather})
    return json.dumps({"error": f"Weather data for {airport_code} not available"})


def find_available_aircraft(airport_code: str) -> str:
    available = [ac for ac in FLEET if ac["location"] == airport_code.upper() and ac["status"] == "available"]
    return json.dumps({"airport": airport_code, "available_aircraft": available})


def check_crew_availability(base: str) -> str:
    available = [c for c in CREW if c["base"] == base.upper() and c["duty_hours_remaining"] > 2.0]
    return json.dumps({"base": base, "available_crew": available})


def propose_recovery_options(flight_number: str, issue: str) -> str:
    flight = next((f for f in FLIGHTS if f["flight"] == flight_number), None)
    if not flight:
        return json.dumps({"error": f"Flight {flight_number} not found"})

    options = []
    if "weather" in issue.lower():
        options.append({"option": "Delay departure until weather improves", "impact": "Passenger inconvenience, crew duty risk"})
        options.append({"option": "Reroute to alternate airport", "impact": "Fuel cost, passenger rebooking"})
    if "mechanical" in issue.lower() or "aircraft" in issue.lower():
        options.append({"option": "Swap aircraft from available fleet", "impact": "Maintenance coordination required"})
        options.append({"option": "Cancel flight and rebook passengers", "impact": "High customer impact"})

    return json.dumps({"flight": flight_number, "options": options})


def estimate_delay_cost(delay_minutes: int, aircraft_type: str = "A320") -> str:
    base_cost_per_min = 75
    if aircraft_type in ("A350", "B787", "B777"):
        base_cost_per_min = 150
    total = delay_minutes * base_cost_per_min
    return json.dumps({
        "delay_minutes": delay_minutes,
        "aircraft_type": aircraft_type,
        "estimated_cost_usd": total,
        "note": "Rough estimate; actual costs vary by airline and route"
    })


TOOL_MAP = {
    "get_flight_status": get_flight_status,
    "check_weather": check_weather,
    "find_available_aircraft": find_available_aircraft,
    "check_crew_availability": check_crew_availability,
    "propose_recovery_options": propose_recovery_options,
    "estimate_delay_cost": estimate_delay_cost,
}


# ============================================================
# 4. SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """You are an AI agent for an Airline Operations Control Center (AOCC).

Your role is to assist operations controllers in managing flight disruptions.
You have access to tools that provide real-time data about flights, weather, aircraft, and crew.

When responding to a disruption:
1. Gather all relevant data using your tools (flight status, weather, available resources)
2. Assess the severity and propagation risk
3. Propose ranked recovery options with cost estimates
4. Always recommend human verification before execution

Guidelines:
- Be concise and operational in your responses
- Prioritize safety and regulatory compliance
- Consider passenger impact alongside operational cost
- Flag when human intervention is required

Current operational context: Normal operations, monitoring for weather disruptions at JFK."""


# ============================================================
# 5. AGENT LOOP (with reference-contract error handling)
# ============================================================

def run_aocc_agent(user_query: str, verbose: bool = True) -> str:
    """
    Run the AOCC agent with tool calling and thinking mode.
    Uses Option (b): reasoning_effort="medium" for multi-step recovery planning.
    """
    if not DEEPSEEK_API_KEY:
        print("❌ CRITICAL FAILURE: API Key not loaded.")
        return ""

    client = OpenAI(
        api_key=DEEPSEEK_API_KEY,
        base_url=DEEPSEEK_BASE_URL,
        timeout=60.0,
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"AOCC CONTROLLER: {user_query}")
        print(f"{'='*60}")

    max_iterations = 10

    for iteration in range(max_iterations):
        for attempt in range(MAX_RETRIES):
            try:
                response = client.chat.completions.create(
                    model=DEEPSEEK_MODEL,
                    messages=messages,
                    tools=TOOLS,
                    reasoning_effort="medium",  # Option (b)
                    stream=False,
                )
                break  # success, exit retry loop

            except AuthenticationError:
                print("\n❌ AUTHENTICATION FAILURE (Error 401): Check your API key.")
                return ""
            except RateLimitError:
                print("\n⚠️ RATE LIMIT FAILURE (Error 429): Too many requests.")
                return ""
            except APIStatusError as e:
                if e.status_code == 402:
                    print("\n⚠️ INSUFFICIENT BALANCE (Error 402): Top up your account.")
                else:
                    print(f"\n❌ API ERROR {e.status_code}: {e}")
                return ""
            except APIConnectionError as e:
                if attempt < MAX_RETRIES - 1:
                    wait_time = 2 ** attempt
                    print(f"❌ CONNECTION FAILURE: {e}. Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"\n❌ CRITICAL FAILURE: Connection failed after {MAX_RETRIES} retries.")
                    return ""
            except Exception as e:
                print(f"\n❌ UNHANDLED ERROR: {e.__class__.__name__}: {e}")
                return ""

        message = response.choices[0].message

        # CRITICAL: Append the FULL assistant message (including reasoning_content)
        # before processing tool calls. Required by DeepSeek thinking mode.
        messages.append(message)

        if message.tool_calls:
            for tool_call in message.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"\n  [Agent calls: {fn_name}({fn_args})]")

                if fn_name in TOOL_MAP:
                    result = TOOL_MAP[fn_name](**fn_args)
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    print(f"  [Result: {result[:200]}{'...' if len(result) > 200 else ''}]")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })
            continue  # loop again to let model process tool results

        # No tool calls — final answer
        final_answer = message.content
        if verbose:
            print(f"\n{'='*60}")
            print(f"AOCC AGENT RESPONSE:")
            print(f"{'='*60}")
            print(final_answer)

        return final_answer

    return "Max iterations reached without final response."


# ============================================================
# 6. DEMO SCENARIOS
# ============================================================

def demo_disruption_scenario():
    print("\n" + "#"*60)
    print("# AOCC AGENT DEMO - DeepSeek V4.1 Flash (reasoning_effort=medium)")
    print("#"*60)

    run_aocc_agent(
        "Flight AA200 is delayed. Check the weather at JFK and ORD, "
        "and recommend recovery options with cost estimates."
    )

    run_aocc_agent(
        "We may need to swap an aircraft for AA100 at JFK. "
        "Check what aircraft and crew are available, and estimate the cost "
        "if we delay the flight by 90 minutes."
    )

    run_aocc_agent(
        "AA300 (LAX-JFK) is showing a mechanical issue. The aircraft type is B737. "
        "Check the flight status, find available aircraft at LAX, and propose "
        "recovery options."
    )


# ============================================================
# 7. MAIN
# ============================================================

if __name__ == "__main__":
    if not DEEPSEEK_API_KEY:
        print("ERROR: DEEPSEEK_API_KEY not found in Colab userdata.")
        print("Add it via Colab sidebar → 🔑 Secrets → DEEPSEEK_API_KEY")
    else:
        demo_disruption_scenario()


############################################################
# AOCC AGENT DEMO - DeepSeek V4.1 Flash (reasoning_effort=medium)
############################################################

AOCC CONTROLLER: Flight AA200 is delayed. Check the weather at JFK and ORD, and recommend recovery options with cost estimates.

  [Agent calls: get_flight_status({'flight_number': 'AA200'})]
  [Result: {"flight": "AA200", "route": "JFK-ORD", "tail": "N202AA", "status": "delayed", "departure": "2026-09-18T15:30"}]

  [Agent calls: check_weather({'airport_code': 'JFK'})]
  [Result: {"airport": "JFK", "condition": "Thunderstorms", "severity": "high", "visibility": "2 miles"}]

  [Agent calls: check_weather({'airport_code': 'ORD'})]
  [Result: {"airport": "ORD", "condition": "Light Rain", "severity": "medium", "visibility": "5 miles"}]

  [Agent calls: find_available_aircraft({'airport_code': 'JFK'})]
  [Result: {"airport": "JFK", "available_aircraft": [{"tail": "N101AA", "type": "A320", "location": "J

## CASE2

In [3]:
import os
import time
import json
from openai import OpenAI
from openai import APIConnectionError, AuthenticationError, RateLimitError, APIStatusError

# --- Configuration (Reference contract from diagnostic script) ---
from google.colab import userdata
DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_MODEL = "deepseek-flash"  # DeepSeek V4.1 Flash
MAX_RETRIES = 3


# ============================================================
# 1. AOCC SIMULATED DATA SOURCES
# ============================================================

FLEET = [
    {"tail": "N101AA", "type": "A320", "location": "JFK", "status": "available"},
    {"tail": "N102AA", "type": "B737", "location": "LAX", "status": "available"},
    {"tail": "N201AA", "type": "A350", "location": "JFK", "status": "maintenance"},
    {"tail": "N202AA", "type": "B787", "location": "ORD", "status": "available"},
    {"tail": "N999AA", "type": "A321", "location": "JFK", "status": "available"},  # True spare
    {"tail": "N998AA", "type": "A320", "location": "LAX", "status": "available"},  # True spare
]

CREW = [
    {"id": "CPT001", "role": "Captain", "base": "JFK", "duty_hours_remaining": 8.5},
    {"id": "FO001", "role": "First Officer", "base": "JFK", "duty_hours_remaining": 9.0},
    {"id": "FA001", "role": "Flight Attendant", "base": "JFK", "duty_hours_remaining": 10.0},
    {"id": "CPT002", "role": "Captain", "base": "LAX", "duty_hours_remaining": 3.0},
    {"id": "FO002", "role": "First Officer", "base": "LAX", "duty_hours_remaining": 7.5},  # Added for LAX
    {"id": "FA002", "role": "Flight Attendant", "base": "LAX", "duty_hours_remaining": 9.0},
]

# Aircraft type now included directly in flight records
FLIGHTS = [
    {"flight": "AA100", "route": "JFK-LAX", "tail": "N101AA", "type": "A320",
     "status": "scheduled", "departure": "2026-09-18T14:00"},
    {"flight": "AA200", "route": "JFK-ORD", "tail": "N202AA", "type": "B787",
     "status": "delayed", "departure": "2026-09-18T15:30"},
    {"flight": "AA300", "route": "LAX-JFK", "tail": "N102AA", "type": "B737",
     "status": "scheduled", "departure": "2026-09-18T16:00"},
]

WEATHER = {
    "JFK": {"condition": "Thunderstorms", "severity": "high", "visibility": "2 miles"},
    "LAX": {"condition": "Clear", "severity": "low", "visibility": "10 miles"},
    "ORD": {"condition": "Light Rain", "severity": "medium", "visibility": "5 miles"},
}


# ============================================================
# 2. TOOL DEFINITIONS
# ============================================================

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_flight_status",
            "description": "Get the current status of a specific flight including delay, aircraft tail, aircraft type, and route.",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {"type": "string", "description": "Flight number, e.g., AA100"},
                },
                "required": ["flight_number"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_aircraft_by_tail",
            "description": "Look up aircraft details by tail number, including type and current location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "tail": {"type": "string", "description": "Aircraft tail number, e.g., N101AA"},
                },
                "required": ["tail"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_weather",
            "description": "Check current weather conditions at an airport.",
            "parameters": {
                "type": "object",
                "properties": {
                    "airport_code": {"type": "string", "description": "IATA airport code, e.g., JFK"},
                },
                "required": ["airport_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "find_available_aircraft",
            "description": "Find available (unassigned) aircraft at a specific airport.",
            "parameters": {
                "type": "object",
                "properties": {
                    "airport_code": {"type": "string", "description": "Airport to search, e.g., JFK"},
                },
                "required": ["airport_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_crew_availability",
            "description": "Check crew availability and duty time remaining at a base airport.",
            "parameters": {
                "type": "object",
                "properties": {
                    "base": {"type": "string", "description": "Crew base airport, e.g., JFK"},
                },
                "required": ["base"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "propose_recovery_options",
            "description": "Generate ranked recovery options for a disrupted flight.",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {"type": "string", "description": "Disrupted flight number"},
                    "issue": {"type": "string", "description": "Description of the disruption"},
                },
                "required": ["flight_number", "issue"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "estimate_delay_cost",
            "description": "Estimate the operational cost of a delay. If aircraft_type is omitted, the tool looks up the type from the flight's tail.",
            "parameters": {
                "type": "object",
                "properties": {
                    "delay_minutes": {"type": "integer", "description": "Delay duration in minutes"},
                    "aircraft_type": {"type": "string", "description": "Aircraft type, e.g., A320. Optional if flight_number is given."},
                    "flight_number": {"type": "string", "description": "Flight number for automatic type lookup. Optional."},
                },
                "required": ["delay_minutes"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "escalate_to_human",
            "description": "Formally escalate a decision to a human controller when the agent cannot safely recommend an action.",
            "parameters": {
                "type": "object",
                "properties": {
                    "reason": {"type": "string", "description": "Why human intervention is required"},
                    "flight_number": {"type": "string", "description": "Related flight, if applicable"},
                },
                "required": ["reason"],
            },
        },
    },
]


# ============================================================
# 3. TOOL IMPLEMENTATIONS
# ============================================================

def get_flight_status(flight_number: str) -> str:
    for f in FLIGHTS:
        if f["flight"] == flight_number:
            return json.dumps(f)
    return json.dumps({"error": f"Flight {flight_number} not found"})


def get_aircraft_by_tail(tail: str) -> str:
    for ac in FLEET:
        if ac["tail"].upper() == tail.upper():
            return json.dumps(ac)
    return json.dumps({"error": f"Aircraft {tail} not found"})


def check_weather(airport_code: str) -> str:
    weather = WEATHER.get(airport_code.upper())
    if weather:
        return json.dumps({"airport": airport_code, **weather})
    return json.dumps({"error": f"Weather data for {airport_code} not available"})


def find_available_aircraft(airport_code: str) -> str:
    """Return only aircraft that are NOT assigned to any scheduled/delayed flight."""
    assigned_tails = {f["tail"] for f in FLIGHTS if f["status"] in ("scheduled", "delayed")}
    available = [
        ac for ac in FLEET
        if ac["location"] == airport_code.upper()
        and ac["status"] == "available"
        and ac["tail"] not in assigned_tails
    ]
    return json.dumps({"airport": airport_code, "available_aircraft": available})


def check_crew_availability(base: str) -> str:
    available = [c for c in CREW if c["base"] == base.upper() and c["duty_hours_remaining"] > 2.0]
    return json.dumps({"base": base, "available_crew": available})


def propose_recovery_options(flight_number: str, issue: str) -> str:
    """Generate recovery options with broadened keyword matching."""
    flight = next((f for f in FLIGHTS if f["flight"] == flight_number), None)
    if not flight:
        return json.dumps({"error": f"Flight {flight_number} not found"})

    issue_lower = issue.lower()
    options = []

    weather_keywords = ["weather", "thunderstorm", "storm", "rain", "snow", "fog",
                        "convective", "wind", "ceiling", "visibility", "ground stop"]
    mech_keywords = ["mechanical", "aircraft", "engine", "hydraulic", "avionics",
                     "mel", "fault", "maintenance", "inspection", "aog"]
    crew_keywords = ["crew", "duty", "rest", "fatigue", "ftl", "captain", "first officer"]

    if any(k in issue_lower for k in weather_keywords):
        options.append({
            "option": "Hold for convective passage",
            "impact": "Passenger inconvenience; recheck weather every 15-30 min",
            "cost_driver": "Delay cost ~$75/min (narrowbody)",
        })
        options.append({
            "option": "Request alternate departure slot / reroute",
            "impact": "Low incremental cost; reduces exposure to flow programs",
            "cost_driver": "Minimal",
        })
        options.append({
            "option": "Reroute to alternate airport",
            "impact": "Fuel cost, passenger rebooking/ground transport",
            "cost_driver": "Fuel + rebooking",
        })

    if any(k in issue_lower for k in mech_keywords):
        options.append({
            "option": "Repair in place",
            "impact": "Maintenance coordination; delay duration unknown until diagnosis",
            "cost_driver": "Delay cost + maintenance labor",
        })
        options.append({
            "option": "Swap aircraft from available fleet",
            "impact": "Maintenance coordination; reconfiguration may be required",
            "cost_driver": "Delay cost + reconfiguration",
        })
        options.append({
            "option": "Cancel and rebook passengers",
            "impact": "High customer impact; last resort",
            "cost_driver": "Rebooking + compensation",
        })

    if any(k in issue_lower for k in crew_keywords):
        options.append({
            "option": "Reserve crew from standby pool",
            "impact": "Depends on standby availability and rest compliance",
            "cost_driver": "Reserve crew premium",
        })
        options.append({
            "option": "Delay until assigned crew becomes legal",
            "impact": "Delay duration driven by FTL rules",
            "cost_driver": "Delay cost",
        })

    if not options:
        options.append({
            "option": "Manual review required",
            "impact": "Issue category not auto-classified — controller assessment needed",
            "cost_driver": "Unknown",
        })

    return json.dumps({"flight": flight_number, "options": options})


def estimate_delay_cost(delay_minutes: int, aircraft_type: str = None, flight_number: str = None) -> str:
    """Estimate delay cost. If aircraft_type omitted, look up from flight_number."""
    resolved_type = aircraft_type

    if not resolved_type and flight_number:
        flight = next((f for f in FLIGHTS if f["flight"] == flight_number), None)
        if flight:
            resolved_type = flight.get("type")

    if not resolved_type:
        resolved_type = "A320"  # fallback
        type_source = "default (no type supplied)"
    else:
        type_source = "resolved" if not aircraft_type else "provided"

    base_cost_per_min = 75
    if resolved_type in ("A350", "B787", "B777", "A330"):
        base_cost_per_min = 150

    total = delay_minutes * base_cost_per_min
    return json.dumps({
        "delay_minutes": delay_minutes,
        "aircraft_type": resolved_type,
        "type_source": type_source,
        "estimated_cost_usd": total,
        "note": "Rough estimate; excludes passenger compensation, misconnects, downstream rotation"
    })


def escalate_to_human(reason: str, flight_number: str = None) -> str:
    """Formal escalation flag for the controller queue."""
    payload = {
        "escalation": True,
        "reason": reason,
        "flight_number": flight_number,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "status": "queued_for_controller_review",
    }
    return json.dumps(payload)


TOOL_MAP = {
    "get_flight_status": get_flight_status,
    "get_aircraft_by_tail": get_aircraft_by_tail,
    "check_weather": check_weather,
    "find_available_aircraft": find_available_aircraft,
    "check_crew_availability": check_crew_availability,
    "propose_recovery_options": propose_recovery_options,
    "estimate_delay_cost": estimate_delay_cost,
    "escalate_to_human": escalate_to_human,
}


# ============================================================
# 4. SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """You are an AI agent for an Airline Operations Control Center (AOCC).

Your role is to assist operations controllers in managing flight disruptions.
You have access to tools that provide real-time data about flights, weather, aircraft, and crew.

When responding to a disruption:
1. Gather all relevant data using your tools (flight status, weather, available resources)
2. Assess the severity and propagation risk
3. Propose ranked recovery options with cost estimates
4. Use escalate_to_human when a decision exceeds your authority or data is insufficient

Guidelines:
- Be concise and operational in your responses
- Prioritize safety and regulatory compliance
- Consider passenger impact alongside operational cost
- When a swap is proposed, verify a TRUE spare exists (not the flight's own tail)
- When estimating cost, use the flight's actual aircraft type via the lookup
- Always recommend human verification before execution

Current operational context: Normal operations, monitoring for weather disruptions at JFK."""


# ============================================================
# 5. AGENT LOOP
# ============================================================

def run_aocc_agent(user_query: str, verbose: bool = True) -> str:
    """Run the AOCC agent with tool calling and thinking mode (reasoning_effort=medium)."""
    if not DEEPSEEK_API_KEY:
        print("❌ CRITICAL FAILURE: API Key not loaded.")
        return ""

    client = OpenAI(
        api_key=DEEPSEEK_API_KEY,
        base_url=DEEPSEEK_BASE_URL,
        timeout=60.0,
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"AOCC CONTROLLER: {user_query}")
        print(f"{'='*60}")

    max_iterations = 10

    for iteration in range(max_iterations):
        # Retry wrapper for transient connection errors
        for attempt in range(MAX_RETRIES):
            try:
                response = client.chat.completions.create(
                    model=DEEPSEEK_MODEL,
                    messages=messages,
                    tools=TOOLS,
                    reasoning_effort="medium",
                    stream=False,
                )
                break

            except AuthenticationError:
                print("\n❌ AUTHENTICATION FAILURE (Error 401): Check your API key.")
                return ""
            except RateLimitError:
                print("\n⚠️ RATE LIMIT FAILURE (Error 429): Too many requests.")
                return ""
            except APIStatusError as e:
                if e.status_code == 402:
                    print("\n⚠️ INSUFFICIENT BALANCE (Error 402): Top up your account.")
                else:
                    print(f"\n❌ API ERROR {e.status_code}: {e}")
                return ""
            except APIConnectionError as e:
                if attempt < MAX_RETRIES - 1:
                    wait_time = 2 ** attempt
                    print(f"❌ CONNECTION FAILURE: {e}. Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"\n❌ CRITICAL FAILURE: Connection failed after {MAX_RETRIES} retries.")
                    return ""
            except Exception as e:
                print(f"\n❌ UNHANDLED ERROR: {e.__class__.__name__}: {e}")
                return ""

        message = response.choices[0].message

        # CRITICAL: append the FULL assistant message (with reasoning_content)
        # before processing tool calls, per DeepSeek thinking-mode requirements.
        messages.append(message)

        if message.tool_calls:
            for tool_call in message.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"\n  [Agent calls: {fn_name}({fn_args})]")

                if fn_name in TOOL_MAP:
                    try:
                        result = TOOL_MAP[fn_name](**fn_args)
                    except TypeError as e:
                        result = json.dumps({"error": f"Bad arguments to {fn_name}: {e}"})
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    preview = result[:200] + ('...' if len(result) > 200 else '')
                    print(f"  [Result: {preview}]")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })
            continue

        # No tool calls — final answer
        final_answer = message.content
        if verbose:
            print(f"\n{'='*60}")
            print(f"AOCC AGENT RESPONSE:")
            print(f"{'='*60}")
            print(final_answer)

        return final_answer

    return "Max iterations reached without final response."


# ============================================================
# 6. DEMO SCENARIOS
# ============================================================

def demo_disruption_scenario():
    print("\n" + "#"*60)
    print("# AOCC AGENT v2 - DeepSeek V4.1 Flash (reasoning_effort=medium)")
    print("#"*60)

    # Scenario 1: Weather disruption — tool should now return populated options
    run_aocc_agent(
        "Flight AA200 is delayed. Check the weather at JFK and ORD, "
        "and recommend recovery options with cost estimates."
    )

    # Scenario 2: Aircraft swap — a true spare (N999AA) now exists at JFK
    run_aocc_agent(
        "We may need to swap an aircraft for AA100 at JFK. "
        "Check what aircraft and crew are available, and estimate the cost "
        "if we delay the flight by 90 minutes."
    )

    # Scenario 3: Mechanical + crew shortage — LAX now has FO002 available
    run_aocc_agent(
        "AA300 (LAX-JFK) is showing a mechanical issue. The aircraft type is B737. "
        "Check the flight status, find available aircraft at LAX, and propose "
        "recovery options."
    )

    # Scenario 4 (new): Scenario that should trigger escalation
    run_aocc_agent(
        "An unclassified sensor anomaly is reported on AA100. "
        "There is no MEL reference and no maintenance disposition available. "
        "What is your recommendation?"
    )


# ============================================================
# 7. MAIN
# ============================================================

if __name__ == "__main__":
    if not DEEPSEEK_API_KEY:
        print("ERROR: DEEPSEEK_API_KEY not found in Colab userdata.")
        print("Add it via Colab sidebar → 🔑 Secrets → DEEPSEEK_API_KEY")
    else:
        demo_disruption_scenario()


############################################################
# AOCC AGENT v2 - DeepSeek V4.1 Flash (reasoning_effort=medium)
############################################################

AOCC CONTROLLER: Flight AA200 is delayed. Check the weather at JFK and ORD, and recommend recovery options with cost estimates.

  [Agent calls: get_flight_status({'flight_number': 'AA200'})]
  [Result: {"flight": "AA200", "route": "JFK-ORD", "tail": "N202AA", "type": "B787", "status": "delayed", "departure": "2026-09-18T15:30"}]

  [Agent calls: check_weather({'airport_code': 'JFK'})]
  [Result: {"airport": "JFK", "condition": "Thunderstorms", "severity": "high", "visibility": "2 miles"}]

  [Agent calls: check_weather({'airport_code': 'ORD'})]
  [Result: {"airport": "ORD", "condition": "Light Rain", "severity": "medium", "visibility": "5 miles"}]

  [Agent calls: get_aircraft_by_tail({'tail': 'N202AA'})]
  [Result: {"tail": "N202AA", "type": "B787", "location": "ORD", "status": "available"}]

  [Age

## CASE3

In [4]:
import os
import time
import json
from openai import OpenAI
from openai import APIConnectionError, AuthenticationError, RateLimitError, APIStatusError

# --- Configuration (Reference contract from diagnostic script) ---
from google.colab import userdata
DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_MODEL = "deepseek-flash"  # DeepSeek V4.1 Flash
MAX_RETRIES = 3


# ============================================================
# 1. AOCC SIMULATED DATA SOURCES
# ============================================================
# NOTE: FLEET.serviceability describes airframe condition ("serviceable" /
# "maintenance"), NOT whether a tail is free. Use find_available_aircraft()
# to determine assignment status, or get_tail_assignments() for a full view.

FLEET = [
    {"tail": "N101AA", "type": "A320", "location": "JFK", "serviceability": "serviceable"},
    {"tail": "N102AA", "type": "B737", "location": "LAX", "serviceability": "serviceable"},
    {"tail": "N201AA", "type": "A350", "location": "JFK", "serviceability": "maintenance"},
    {"tail": "N202AA", "type": "B787", "location": "JFK", "serviceability": "serviceable"},  # FIX: moved from ORD to JFK
    {"tail": "N999AA", "type": "A321", "location": "JFK", "serviceability": "serviceable"},
    {"tail": "N998AA", "type": "A320", "location": "LAX", "serviceability": "serviceable"},
]

CREW = [
    {"id": "CPT001", "role": "Captain", "base": "JFK", "type_ratings": ["A320", "A321"], "duty_hours_remaining": 8.5},
    {"id": "FO001", "role": "First Officer", "base": "JFK", "type_ratings": ["A320", "A321"], "duty_hours_remaining": 9.0},
    {"id": "FA001", "role": "Flight Attendant", "base": "JFK", "type_ratings": ["A320", "A321", "B737", "B787"], "duty_hours_remaining": 10.0},
    {"id": "CPT002", "role": "Captain", "base": "LAX", "type_ratings": ["B737"], "duty_hours_remaining": 3.0},
    {"id": "FO002", "role": "First Officer", "base": "LAX", "type_ratings": ["B737"], "duty_hours_remaining": 7.5},
    {"id": "FA002", "role": "Flight Attendant", "base": "LAX", "type_ratings": ["A320", "A321", "B737", "B787"], "duty_hours_remaining": 9.0},
]

FLIGHTS = [
    {"flight": "AA100", "route": "JFK-LAX", "tail": "N101AA", "type": "A320",
     "status": "scheduled", "departure": "2026-09-18T14:00"},
    {"flight": "AA200", "route": "JFK-ORD", "tail": "N202AA", "type": "B787",
     "status": "delayed", "departure": "2026-09-18T15:30"},
    {"flight": "AA300", "route": "LAX-JFK", "tail": "N102AA", "type": "B737",
     "status": "scheduled", "departure": "2026-09-18T16:00"},
]

WEATHER = {
    "JFK": {"condition": "Thunderstorms", "severity": "high", "visibility": "2 miles"},
    "LAX": {"condition": "Clear", "severity": "low", "visibility": "10 miles"},
    "ORD": {"condition": "Light Rain", "severity": "medium", "visibility": "5 miles"},
}


# ============================================================
# 2. TOOL DEFINITIONS
# ============================================================

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_flight_status",
            "description": "Get the current status of a specific flight including delay, tail, type, and route.",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {"type": "string", "description": "Flight number, e.g., AA100"},
                },
                "required": ["flight_number"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_aircraft_by_tail",
            "description": "Look up aircraft details by tail number, including type, current location, and serviceability status. NOTE: serviceability does not indicate whether the tail is assigned to a flight.",
            "parameters": {
                "type": "object",
                "properties": {
                    "tail": {"type": "string", "description": "Aircraft tail number, e.g., N101AA"},
                },
                "required": ["tail"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_tail_assignments",
            "description": "List all tails physically present at an airport with their assignment status (assigned flight or unassigned spare). Use this to distinguish a true spare from a tail already committed to another flight.",
            "parameters": {
                "type": "object",
                "properties": {
                    "airport_code": {"type": "string", "description": "IATA airport code, e.g., JFK"},
                },
                "required": ["airport_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_weather",
            "description": "Check current weather conditions at an airport.",
            "parameters": {
                "type": "object",
                "properties": {
                    "airport_code": {"type": "string", "description": "IATA airport code, e.g., JFK"},
                },
                "required": ["airport_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "find_available_aircraft",
            "description": "Find true spare aircraft at a specific airport (serviceable AND not assigned to any flight).",
            "parameters": {
                "type": "object",
                "properties": {
                    "airport_code": {"type": "string", "description": "Airport to search, e.g., JFK"},
                },
                "required": ["airport_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_crew_availability",
            "description": "Check crew availability, duty time remaining, and type ratings at a base airport. Optionally filter by required aircraft type.",
            "parameters": {
                "type": "object",
                "properties": {
                    "base": {"type": "string", "description": "Crew base airport, e.g., JFK"},
                    "aircraft_type": {"type": "string", "description": "Optional: filter crew by type rating, e.g., A321"},
                },
                "required": ["base"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "propose_recovery_options",
            "description": "Generate ranked recovery options for a disrupted flight.",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {"type": "string", "description": "Disrupted flight number"},
                    "issue": {"type": "string", "description": "Description of the disruption"},
                },
                "required": ["flight_number", "issue"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "estimate_delay_cost",
            "description": "Estimate the operational cost of a delay. If aircraft_type is omitted, the tool resolves it from the flight's tail.",
            "parameters": {
                "type": "object",
                "properties": {
                    "delay_minutes": {"type": "integer", "description": "Delay duration in minutes"},
                    "aircraft_type": {"type": "string", "description": "Optional: aircraft type, e.g., A320."},
                    "flight_number": {"type": "string", "description": "Optional: flight number for automatic type lookup."},
                },
                "required": ["delay_minutes"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "escalate_to_human",
            "description": "Formally escalate a decision to a human controller when the agent cannot safely recommend an action.",
            "parameters": {
                "type": "object",
                "properties": {
                    "reason": {"type": "string", "description": "Why human intervention is required"},
                    "flight_number": {"type": "string", "description": "Related flight, if applicable"},
                },
                "required": ["reason"],
            },
        },
    },
]


# ============================================================
# 3. TOOL IMPLEMENTATIONS
# ============================================================

def get_flight_status(flight_number: str) -> str:
    for f in FLIGHTS:
        if f["flight"] == flight_number:
            return json.dumps(f)
    return json.dumps({"error": f"Flight {flight_number} not found"})


def get_aircraft_by_tail(tail: str) -> str:
    for ac in FLEET:
        if ac["tail"].upper() == tail.upper():
            return json.dumps(ac)
    return json.dumps({"error": f"Aircraft {tail} not found"})


def get_tail_assignments(airport_code: str) -> str:
    """List all tails physically at an airport, with their assignment status."""
    airport = airport_code.upper()
    assigned_map = {f["tail"]: f["flight"] for f in FLIGHTS if f["status"] in ("scheduled", "delayed")}

    result = []
    for ac in FLEET:
        if ac["location"] != airport:
            continue
        entry = {
            "tail": ac["tail"],
            "type": ac["type"],
            "serviceability": ac["serviceability"],
            "assignment": assigned_map.get(ac["tail"], None),  # None = unassigned
        }
        result.append(entry)

    return json.dumps({"airport": airport, "tails": result})


def check_weather(airport_code: str) -> str:
    weather = WEATHER.get(airport_code.upper())
    if weather:
        return json.dumps({"airport": airport_code, **weather})
    return json.dumps({"error": f"Weather data for {airport_code} not available"})


def find_available_aircraft(airport_code: str) -> str:
    """Return only serviceable aircraft NOT assigned to any flight."""
    assigned_tails = {f["tail"] for f in FLIGHTS if f["status"] in ("scheduled", "delayed")}
    available = [
        ac for ac in FLEET
        if ac["location"] == airport_code.upper()
        and ac["serviceability"] == "serviceable"
        and ac["tail"] not in assigned_tails
    ]
    return json.dumps({"airport": airport_code, "available_aircraft": available})


def check_crew_availability(base: str, aircraft_type: str = None) -> str:
    available = [
        c for c in CREW
        if c["base"] == base.upper()
        and c["duty_hours_remaining"] > 2.0
        and (aircraft_type is None or aircraft_type in c.get("type_ratings", []))
    ]
    return json.dumps({"base": base, "filter_type": aircraft_type, "available_crew": available})


def propose_recovery_options(flight_number: str, issue: str) -> str:
    flight = next((f for f in FLIGHTS if f["flight"] == flight_number), None)
    if not flight:
        return json.dumps({"error": f"Flight {flight_number} not found"})

    issue_lower = issue.lower()
    options = []

    weather_keywords = ["weather", "thunderstorm", "storm", "rain", "snow", "fog",
                        "convective", "wind", "ceiling", "visibility", "ground stop"]
    mech_keywords = ["mechanical", "aircraft", "engine", "hydraulic", "avionics",
                     "mel", "fault", "maintenance", "inspection", "aog"]
    crew_keywords = ["crew", "duty", "rest", "fatigue", "ftl", "captain", "first officer"]

    if any(k in issue_lower for k in weather_keywords):
        options.append({"option": "Hold for convective passage",
                        "impact": "Passenger inconvenience; recheck weather every 15-30 min",
                        "cost_driver": "Delay cost"})
        options.append({"option": "Request alternate departure slot / reroute",
                        "impact": "Low incremental cost; reduces exposure to flow programs",
                        "cost_driver": "Minimal"})
        options.append({"option": "Reroute to alternate airport",
                        "impact": "Fuel cost, passenger rebooking/ground transport",
                        "cost_driver": "Fuel + rebooking"})

    if any(k in issue_lower for k in mech_keywords):
        options.append({"option": "Repair in place",
                        "impact": "Maintenance coordination; delay duration unknown until diagnosis",
                        "cost_driver": "Delay cost + maintenance labor"})
        options.append({"option": "Swap aircraft from available fleet",
                        "impact": "Maintenance coordination; reconfiguration may be required",
                        "cost_driver": "Delay cost + reconfiguration"})
        options.append({"option": "Cancel and rebook passengers",
                        "impact": "High customer impact; last resort",
                        "cost_driver": "Rebooking + compensation"})

    if any(k in issue_lower for k in crew_keywords):
        options.append({"option": "Reserve crew from standby pool",
                        "impact": "Depends on standby availability and rest compliance",
                        "cost_driver": "Reserve crew premium"})
        options.append({"option": "Delay until assigned crew becomes legal",
                        "impact": "Delay duration driven by FTL rules",
                        "cost_driver": "Delay cost"})

    if not options:
        options.append({"option": "Manual review required",
                        "impact": "Issue category not auto-classified — controller assessment needed",
                        "cost_driver": "Unknown"})

    return json.dumps({"flight": flight_number, "options": options})


def estimate_delay_cost(delay_minutes: int, aircraft_type: str = None, flight_number: str = None) -> str:
    resolved_type = aircraft_type
    type_source = "provided"

    if not resolved_type and flight_number:
        flight = next((f for f in FLIGHTS if f["flight"] == flight_number), None)
        if flight:
            resolved_type = flight.get("type")
            type_source = "resolved_from_flight"

    if not resolved_type:
        resolved_type = "A320"
        type_source = "default_fallback"

    base_cost_per_min = 75
    if resolved_type in ("A350", "B787", "B777", "A330"):
        base_cost_per_min = 150

    total = delay_minutes * base_cost_per_min
    return json.dumps({
        "delay_minutes": delay_minutes,
        "aircraft_type": resolved_type,
        "type_source": type_source,
        "estimated_cost_usd": total,
        "note": "Rough estimate; excludes passenger compensation, misconnects, downstream rotation"
    })


def escalate_to_human(reason: str, flight_number: str = None) -> str:
    return json.dumps({
        "escalation": True,
        "reason": reason,
        "flight_number": flight_number,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "status": "queued_for_controller_review",
    })


TOOL_MAP = {
    "get_flight_status": get_flight_status,
    "get_aircraft_by_tail": get_aircraft_by_tail,
    "get_tail_assignments": get_tail_assignments,
    "check_weather": check_weather,
    "find_available_aircraft": find_available_aircraft,
    "check_crew_availability": check_crew_availability,
    "propose_recovery_options": propose_recovery_options,
    "estimate_delay_cost": estimate_delay_cost,
    "escalate_to_human": escalate_to_human,
}


# ============================================================
# 4. SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """You are an AI agent for an Airline Operations Control Center (AOCC).

Your role is to assist operations controllers in managing flight disruptions.
You have access to tools that provide real-time data about flights, weather, aircraft, and crew.

When responding to a disruption:
1. Gather all relevant data using your tools (flight status, weather, available resources)
2. Assess the severity and propagation risk
3. Propose ranked recovery options with cost estimates
4. Use escalate_to_human when a decision exceeds your authority or data is insufficient

Guidelines:
- Be concise and operational in your responses
- Prioritize safety and regulatory compliance
- Consider passenger impact alongside operational cost
- When a swap is proposed, verify a TRUE spare exists via find_available_aircraft
  and check type compatibility (widebody vs narrowbody downgauge requires authority sign-off)
- When checking crew for a swap, filter by the replacement aircraft's type rating
- When estimating cost, use the flight's actual aircraft type via flight_number lookup
- Do NOT run more than TWO cost scenarios per flight unless the controller asks for more
- Always recommend human verification before execution

Data model notes:
- FLEET.serviceability indicates airframe condition only (serviceable/maintenance).
  It does NOT mean the tail is unassigned. Use find_available_aircraft or
  get_tail_assignments to determine true availability.

Current operational context: Normal operations, monitoring for weather disruptions at JFK."""


# ============================================================
# 5. AGENT LOOP
# ============================================================

def run_aocc_agent(user_query: str, verbose: bool = True) -> str:
    if not DEEPSEEK_API_KEY:
        print("❌ CRITICAL FAILURE: API Key not loaded.")
        return ""

    client = OpenAI(
        api_key=DEEPSEEK_API_KEY,
        base_url=DEEPSEEK_BASE_URL,
        timeout=60.0,
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"AOCC CONTROLLER: {user_query}")
        print(f"{'='*60}")

    max_iterations = 10

    for iteration in range(max_iterations):
        for attempt in range(MAX_RETRIES):
            try:
                response = client.chat.completions.create(
                    model=DEEPSEEK_MODEL,
                    messages=messages,
                    tools=TOOLS,
                    reasoning_effort="medium",
                    stream=False,
                )
                break

            except AuthenticationError:
                print("\n❌ AUTHENTICATION FAILURE (Error 401): Check your API key.")
                return ""
            except RateLimitError:
                print("\n⚠️ RATE LIMIT FAILURE (Error 429): Too many requests.")
                return ""
            except APIStatusError as e:
                if e.status_code == 402:
                    print("\n⚠️ INSUFFICIENT BALANCE (Error 402): Top up your account.")
                else:
                    print(f"\n❌ API ERROR {e.status_code}: {e}")
                return ""
            except APIConnectionError as e:
                if attempt < MAX_RETRIES - 1:
                    wait_time = 2 ** attempt
                    print(f"❌ CONNECTION FAILURE: {e}. Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"\n❌ CRITICAL FAILURE: Connection failed after {MAX_RETRIES} retries.")
                    return ""
            except Exception as e:
                print(f"\n❌ UNHANDLED ERROR: {e.__class__.__name__}: {e}")
                return ""

        message = response.choices[0].message
        messages.append(message)  # Required for thinking mode

        if message.tool_calls:
            for tool_call in message.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"\n  [Agent calls: {fn_name}({fn_args})]")

                if fn_name in TOOL_MAP:
                    try:
                        result = TOOL_MAP[fn_name](**fn_args)
                    except TypeError as e:
                        result = json.dumps({"error": f"Bad arguments to {fn_name}: {e}"})
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    preview = result[:200] + ('...' if len(result) > 200 else '')
                    print(f"  [Result: {preview}]")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })
            continue

        final_answer = message.content
        if verbose:
            print(f"\n{'='*60}")
            print(f"AOCC AGENT RESPONSE:")
            print(f"{'='*60}")
            print(final_answer)

        return final_answer

    return "Max iterations reached without final response."


# ============================================================
# 6. DEMO SCENARIOS
# ============================================================

def demo_disruption_scenario():
    print("\n" + "#"*60)
    print("# AOCC AGENT v3 - DeepSeek V4.1 Flash (reasoning_effort=medium)")
    print("#"*60)

    # Scenario 1: Weather — no false anomaly now (N202AA at JFK)
    run_aocc_agent(
        "Flight AA200 is delayed. Check the weather at JFK and ORD, "
        "and recommend recovery options with cost estimates."
    )

    # Scenario 2: Real disruption trigger for the swap
    run_aocc_agent(
        "AA100 has a hydraulic leak discovered at the gate at JFK and needs an "
        "aircraft swap. Check available aircraft and crew (filter by the replacement "
        "type), and estimate the cost if we delay the flight by 90 minutes instead of swapping."
    )

    # Scenario 3: Mechanical + crew duty constraint
    run_aocc_agent(
        "AA300 (LAX-JFK) is showing a mechanical issue. The aircraft type is B737. "
        "Check the flight status, find available aircraft at LAX, and propose "
        "recovery options."
    )

    # Scenario 4: Escalation trigger — unclassified fault
    run_aocc_agent(
        "An unclassified sensor anomaly is reported on AA100. "
        "There is no MEL reference and no maintenance disposition available. "
        "What is your recommendation?"
    )


# ============================================================
# 7. MAIN
# ============================================================

if __name__ == "__main__":
    if not DEEPSEEK_API_KEY:
        print("ERROR: DEEPSEEK_API_KEY not found in Colab userdata.")
        print("Add it via Colab sidebar → 🔑 Secrets → DEEPSEEK_API_KEY")
    else:
        demo_disruption_scenario()


############################################################
# AOCC AGENT v3 - DeepSeek V4.1 Flash (reasoning_effort=medium)
############################################################

AOCC CONTROLLER: Flight AA200 is delayed. Check the weather at JFK and ORD, and recommend recovery options with cost estimates.

  [Agent calls: get_flight_status({'flight_number': 'AA200'})]
  [Result: {"flight": "AA200", "route": "JFK-ORD", "tail": "N202AA", "type": "B787", "status": "delayed", "departure": "2026-09-18T15:30"}]

  [Agent calls: check_weather({'airport_code': 'JFK'})]
  [Result: {"airport": "JFK", "condition": "Thunderstorms", "severity": "high", "visibility": "2 miles"}]

  [Agent calls: check_weather({'airport_code': 'ORD'})]
  [Result: {"airport": "ORD", "condition": "Light Rain", "severity": "medium", "visibility": "5 miles"}]

  [Agent calls: propose_recovery_options({'flight_number': 'AA200', 'issue': 'Delayed departure at JFK (thunderstorms, high severity, 2mi visibility) to O

## CASE4

In [5]:
import os
import time
import json
from openai import OpenAI
from openai import APIConnectionError, AuthenticationError, RateLimitError, APIStatusError

# --- Configuration (Reference contract from diagnostic script) ---
from google.colab import userdata
DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_MODEL = "deepseek-flash"  # DeepSeek V4.1 Flash
MAX_RETRIES = 3


# ============================================================
# 1. AOCC SIMULATED DATA SOURCES
# ============================================================
# NOTE: FLEET.serviceability = airframe condition only. Assignment is derived
# from FLIGHTS. Use find_available_aircraft() or get_tail_assignments() to
# determine true spare status.

FLEET = [
    {"tail": "N101AA", "type": "A320", "location": "JFK", "serviceability": "maintenance"},  # AA100 AOG
    {"tail": "N102AA", "type": "B737", "location": "LAX", "serviceability": "serviceable"},
    {"tail": "N201AA", "type": "A350", "location": "JFK", "serviceability": "maintenance"},
    {"tail": "N202AA", "type": "B787", "location": "JFK", "serviceability": "maintenance"},  # AA200 AOG
    {"tail": "N999AA", "type": "A321", "location": "JFK", "serviceability": "serviceable"},  # ONLY SPARE
    {"tail": "N998AA", "type": "A320", "location": "LAX", "serviceability": "serviceable"},
]

CREW = [
    {"id": "CPT001", "role": "Captain", "base": "JFK", "type_ratings": ["A320", "A321"], "duty_hours_remaining": 8.5},
    {"id": "FO001", "role": "First Officer", "base": "JFK", "type_ratings": ["A320", "A321"], "duty_hours_remaining": 9.0},
    {"id": "FA001", "role": "Flight Attendant", "base": "JFK", "type_ratings": ["A320", "A321", "B737", "B787"], "duty_hours_remaining": 10.0},
    {"id": "CPT002", "role": "Captain", "base": "LAX", "type_ratings": ["B737"], "duty_hours_remaining": 3.0},
    {"id": "FO002", "role": "First Officer", "base": "LAX", "type_ratings": ["B737"], "duty_hours_remaining": 7.5},
    {"id": "FA002", "role": "Flight Attendant", "base": "LAX", "type_ratings": ["A320", "A321", "B737", "B787"], "duty_hours_remaining": 9.0},
]

FLIGHTS = [
    {"flight": "AA100", "route": "JFK-LAX", "tail": "N101AA", "type": "A320",
     "status": "AOG", "departure": "2026-09-18T14:00", "pax_load": 178, "priority": "high"},
    {"flight": "AA200", "route": "JFK-ORD", "tail": "N202AA", "type": "B787",
     "status": "AOG", "departure": "2026-09-18T15:30", "pax_load": 240, "priority": "medium"},
    {"flight": "AA300", "route": "LAX-JFK", "tail": "N102AA", "type": "B737",
     "status": "scheduled", "departure": "2026-09-18T16:00", "pax_load": 160, "priority": "low"},
]

WEATHER = {
    "JFK": {"condition": "Thunderstorms", "severity": "high", "visibility": "2 miles"},
    "LAX": {"condition": "Clear", "severity": "low", "visibility": "10 miles"},
    "ORD": {"condition": "Light Rain", "severity": "medium", "visibility": "5 miles"},
}


# ============================================================
# 2. TOOL DEFINITIONS
# ============================================================

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_flight_status",
            "description": "Get the current status of a specific flight including delay, tail, type, route, pax load, and priority.",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {"type": "string", "description": "Flight number, e.g., AA100"},
                },
                "required": ["flight_number"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_aircraft_by_tail",
            "description": "Look up aircraft details by tail number, including type, current location, and serviceability. NOTE: serviceability does not indicate whether the tail is assigned to a flight.",
            "parameters": {
                "type": "object",
                "properties": {
                    "tail": {"type": "string", "description": "Aircraft tail number, e.g., N101AA"},
                },
                "required": ["tail"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_tail_assignments",
            "description": "List all tails physically present at an airport with their assignment status (assigned flight or unassigned spare).",
            "parameters": {
                "type": "object",
                "properties": {
                    "airport_code": {"type": "string", "description": "IATA airport code, e.g., JFK"},
                },
                "required": ["airport_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_weather",
            "description": "Check current weather conditions at an airport.",
            "parameters": {
                "type": "object",
                "properties": {
                    "airport_code": {"type": "string", "description": "IATA airport code, e.g., JFK"},
                },
                "required": ["airport_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "find_available_aircraft",
            "description": "Find true spare aircraft at a specific airport (serviceable AND not assigned to any flight).",
            "parameters": {
                "type": "object",
                "properties": {
                    "airport_code": {"type": "string", "description": "Airport to search, e.g., JFK"},
                },
                "required": ["airport_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_crew_availability",
            "description": "Check crew availability, duty time remaining, and type ratings at a base airport. Optionally filter by required aircraft type.",
            "parameters": {
                "type": "object",
                "properties": {
                    "base": {"type": "string", "description": "Crew base airport, e.g., JFK"},
                    "aircraft_type": {"type": "string", "description": "Optional: filter crew by type rating, e.g., A321"},
                },
                "required": ["base"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_passenger_load",
            "description": "Get passenger count and priority for a flight. Use this when comparing flights for resource allocation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {"type": "string", "description": "Flight number, e.g., AA100"},
                },
                "required": ["flight_number"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "resolve_resource_conflict",
            "description": "When multiple flights claim the same spare resource, apply the conflict-resolution policy and return a ranked allocation. MUST be used whenever 2+ flights claim one tail.",
            "parameters": {
                "type": "object",
                "properties": {
                    "resource_tail": {"type": "string", "description": "The contested tail number"},
                    "claiming_flights": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of flight numbers claiming the resource",
                    },
                },
                "required": ["resource_tail", "claiming_flights"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "propose_recovery_options",
            "description": "Generate ranked recovery options for a disrupted flight.",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {"type": "string", "description": "Disrupted flight number"},
                    "issue": {"type": "string", "description": "Description of the disruption"},
                },
                "required": ["flight_number", "issue"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "estimate_delay_cost",
            "description": "Estimate the operational cost of a delay. If aircraft_type is omitted, the tool resolves it from the flight's tail.",
            "parameters": {
                "type": "object",
                "properties": {
                    "delay_minutes": {"type": "integer", "description": "Delay duration in minutes"},
                    "aircraft_type": {"type": "string", "description": "Optional: aircraft type, e.g., A320."},
                    "flight_number": {"type": "string", "description": "Optional: flight number for automatic type lookup."},
                },
                "required": ["delay_minutes"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "escalate_to_human",
            "description": "Formally escalate a decision to a human controller when the agent cannot safely recommend an action.",
            "parameters": {
                "type": "object",
                "properties": {
                    "reason": {"type": "string", "description": "Why human intervention is required"},
                    "flight_number": {"type": "string", "description": "Related flight, if applicable"},
                },
                "required": ["reason"],
            },
        },
    },
]


# ============================================================
# 3. TOOL IMPLEMENTATIONS
# ============================================================

def get_flight_status(flight_number: str) -> str:
    for f in FLIGHTS:
        if f["flight"] == flight_number:
            return json.dumps(f)
    return json.dumps({"error": f"Flight {flight_number} not found"})


def get_aircraft_by_tail(tail: str) -> str:
    for ac in FLEET:
        if ac["tail"].upper() == tail.upper():
            return json.dumps(ac)
    return json.dumps({"error": f"Aircraft {tail} not found"})


def get_tail_assignments(airport_code: str) -> str:
    airport = airport_code.upper()
    assigned_map = {f["tail"]: f["flight"] for f in FLIGHTS if f["status"] in ("scheduled", "delayed", "AOG")}

    result = []
    for ac in FLEET:
        if ac["location"] != airport:
            continue
        result.append({
            "tail": ac["tail"],
            "type": ac["type"],
            "serviceability": ac["serviceability"],
            "assignment": assigned_map.get(ac["tail"], None),
        })

    return json.dumps({"airport": airport, "tails": result})


def check_weather(airport_code: str) -> str:
    weather = WEATHER.get(airport_code.upper())
    if weather:
        return json.dumps({"airport": airport_code, **weather})
    return json.dumps({"error": f"Weather data for {airport_code} not available"})


def find_available_aircraft(airport_code: str) -> str:
    assigned_tails = {f["tail"] for f in FLIGHTS if f["status"] in ("scheduled", "delayed", "AOG")}
    available = [
        ac for ac in FLEET
        if ac["location"] == airport_code.upper()
        and ac["serviceability"] == "serviceable"
        and ac["tail"] not in assigned_tails
    ]
    return json.dumps({"airport": airport_code, "available_aircraft": available})


def check_crew_availability(base: str, aircraft_type: str = None) -> str:
    available = [
        c for c in CREW
        if c["base"] == base.upper()
        and c["duty_hours_remaining"] > 2.0
        and (aircraft_type is None or aircraft_type in c.get("type_ratings", []))
    ]
    return json.dumps({"base": base, "filter_type": aircraft_type, "available_crew": available})


def get_passenger_load(flight_number: str) -> str:
    flight = next((f for f in FLIGHTS if f["flight"] == flight_number), None)
    if not flight:
        return json.dumps({"error": f"Flight {flight_number} not found"})
    return json.dumps({
        "flight": flight_number,
        "pax_load": flight.get("pax_load", 0),
        "priority": flight.get("priority", "unknown"),
        "note": "Passenger load affects rebooking cost and compensation liability",
    })


def resolve_resource_conflict(resource_tail: str, claiming_flights: list) -> str:
    """
    Apply AOCC conflict-resolution policy.

    Policy (v4 - revised for departure urgency):
      1. AOG status (all claims in this demo are AOG, so this tier usually ties)
      2. Departure time (asc) — earliest departure gets priority
      3. Passenger load (desc) — tiebreaker
      4. Priority (high > medium > low) — final tiebreaker

    Rationale: Earlier departures have less schedule slack; delaying them
    cascades further. Load is a secondary consideration once urgency is equal.
    """
    claims = []
    for fn in claiming_flights:
        flight = next((f for f in FLIGHTS if f["flight"] == fn), None)
        if not flight:
            continue
        claims.append({
            "flight": fn,
            "status": flight["status"],
            "pax_load": flight.get("pax_load", 0),
            "priority": flight.get("priority", "unknown"),
            "departure": flight["departure"],
            "type": flight["type"],
        })

    priority_rank = {"high": 0, "medium": 1, "low": 2, "unknown": 3}

    def rank_key(c):
        aog_score = 0 if c["status"] == "AOG" else 1
        return (
            aog_score,                       # AOG first
            c["departure"],                  # earlier departure wins
            -c["pax_load"],                  # higher load tiebreaker
            priority_rank.get(c["priority"], 3),
        )

    claims.sort(key=rank_key)

    winner = claims[0] if claims else None
    losers = claims[1:] if len(claims) > 1 else []

    return json.dumps({
        "resource": resource_tail,
        "winner": winner,
        "losers": losers,
        "policy": "AOG > departure_urgency (asc) > pax_load (desc) > priority",
        "rationale": "All claims AOG; allocation driven by schedule urgency first, passenger impact as tiebreaker.",
    })


def propose_recovery_options(flight_number: str, issue: str) -> str:
    flight = next((f for f in FLIGHTS if f["flight"] == flight_number), None)
    if not flight:
        return json.dumps({"error": f"Flight {flight_number} not found"})

    issue_lower = issue.lower()
    options = []

    weather_keywords = ["weather", "thunderstorm", "storm", "rain", "snow", "fog",
                        "convective", "wind", "ceiling", "visibility", "ground stop"]
    mech_keywords = ["mechanical", "aircraft", "engine", "hydraulic", "avionics",
                     "mel", "fault", "maintenance", "inspection", "aog"]
    crew_keywords = ["crew", "duty", "rest", "fatigue", "ftl", "captain", "first officer"]

    if any(k in issue_lower for k in weather_keywords):
        options.append({"option": "Hold for convective passage",
                        "impact": "Passenger inconvenience; recheck weather every 15-30 min",
                        "cost_driver": "Delay cost"})
        options.append({"option": "Request alternate departure slot / reroute",
                        "impact": "Low incremental cost; reduces exposure to flow programs",
                        "cost_driver": "Minimal"})
        options.append({"option": "Reroute to alternate airport",
                        "impact": "Fuel cost, passenger rebooking/ground transport",
                        "cost_driver": "Fuel + rebooking"})

    if any(k in issue_lower for k in mech_keywords):
        options.append({"option": "Repair in place",
                        "impact": "Maintenance coordination; delay duration unknown until diagnosis",
                        "cost_driver": "Delay cost + maintenance labor"})
        options.append({"option": "Swap aircraft from available fleet",
                        "impact": "Maintenance coordination; reconfiguration may be required",
                        "cost_driver": "Delay cost + reconfiguration"})
        options.append({"option": "Cancel and rebook passengers",
                        "impact": "High customer impact; last resort",
                        "cost_driver": "Rebooking + compensation"})

    if any(k in issue_lower for k in crew_keywords):
        options.append({"option": "Reserve crew from standby pool",
                        "impact": "Depends on standby availability and rest compliance",
                        "cost_driver": "Reserve crew premium"})
        options.append({"option": "Delay until assigned crew becomes legal",
                        "impact": "Delay duration driven by FTL rules",
                        "cost_driver": "Delay cost"})

    if not options:
        options.append({"option": "Manual review required",
                        "impact": "Issue category not auto-classified — controller assessment needed",
                        "cost_driver": "Unknown"})

    return json.dumps({"flight": flight_number, "options": options})


def estimate_delay_cost(delay_minutes: int, aircraft_type: str = None, flight_number: str = None) -> str:
    resolved_type = aircraft_type
    type_source = "provided"

    if not resolved_type and flight_number:
        flight = next((f for f in FLIGHTS if f["flight"] == flight_number), None)
        if flight:
            resolved_type = flight.get("type")
            type_source = "resolved_from_flight"

    if not resolved_type:
        resolved_type = "A320"
        type_source = "default_fallback"

    base_cost_per_min = 75
    if resolved_type in ("A350", "B787", "B777", "A330"):
        base_cost_per_min = 150

    total = delay_minutes * base_cost_per_min
    return json.dumps({
        "delay_minutes": delay_minutes,
        "aircraft_type": resolved_type,
        "type_source": type_source,
        "estimated_cost_usd": total,
        "note": "Rough estimate; excludes passenger compensation, misconnects, downstream rotation"
    })


def escalate_to_human(reason: str, flight_number: str = None) -> str:
    return json.dumps({
        "escalation": True,
        "reason": reason,
        "flight_number": flight_number,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "status": "queued_for_controller_review",
    })


TOOL_MAP = {
    "get_flight_status": get_flight_status,
    "get_aircraft_by_tail": get_aircraft_by_tail,
    "get_tail_assignments": get_tail_assignments,
    "check_weather": check_weather,
    "find_available_aircraft": find_available_aircraft,
    "check_crew_availability": check_crew_availability,
    "get_passenger_load": get_passenger_load,
    "resolve_resource_conflict": resolve_resource_conflict,
    "propose_recovery_options": propose_recovery_options,
    "estimate_delay_cost": estimate_delay_cost,
    "escalate_to_human": escalate_to_human,
}


# ============================================================
# 4. SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """You are an AI agent for an Airline Operations Control Center (AOCC).

Your role is to assist operations controllers in managing flight disruptions.
You have access to tools that provide real-time data about flights, weather, aircraft, and crew.

When responding to a disruption:
1. Gather all relevant data using your tools (flight status, weather, available resources)
2. Assess the severity and propagation risk
3. Propose ranked recovery options with cost estimates
4. Use escalate_to_human when a decision exceeds your authority or data is insufficient

CONFLICT RESOLUTION (critical):
- When TWO OR MORE flights claim the same spare resource, you MUST call resolve_resource_conflict.
- NEVER allocate a single tail to two flights.
- The tool returns a winner and losers based on policy: AOG > departure urgency > pax load > priority.
- Document the losing flight's path forward (delay, cancel, source alternate resource, escalate).
- If the losing flight's outcome is unacceptable (e.g., cancellation without alternatives), escalate to human.

Guidelines:
- Be concise and operational in your responses
- Prioritize safety and regulatory compliance
- Consider passenger impact alongside operational cost
- When a swap is proposed, verify a TRUE spare exists via find_available_aircraft
  and check type compatibility (widebody vs narrowbody downgauge requires authority sign-off)
- When checking crew for a swap, filter by the replacement aircraft's type rating
- When estimating cost, use the flight's actual aircraft type via flight_number lookup
- Do NOT run more than TWO cost scenarios per flight unless the controller asks for more
- Always recommend human verification before execution

Data model notes:
- FLEET.serviceability indicates airframe condition only (serviceable/maintenance).
  It does NOT mean the tail is unassigned. Use find_available_aircraft or
  get_tail_assignments to determine true availability.

Current operational context: JFK has a single serviceable unassigned spare (N999AA, A321).
Two flights (AA100, AA200) are AOG at JFK. Resource contention expected."""


# ============================================================
# 5. AGENT LOOP
# ============================================================

def run_aocc_agent(user_query: str, verbose: bool = True) -> str:
    if not DEEPSEEK_API_KEY:
        print("❌ CRITICAL FAILURE: API Key not loaded.")
        return ""

    client = OpenAI(
        api_key=DEEPSEEK_API_KEY,
        base_url=DEEPSEEK_BASE_URL,
        timeout=60.0,
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"AOCC CONTROLLER: {user_query}")
        print(f"{'='*60}")

    max_iterations = 12

    for iteration in range(max_iterations):
        for attempt in range(MAX_RETRIES):
            try:
                response = client.chat.completions.create(
                    model=DEEPSEEK_MODEL,
                    messages=messages,
                    tools=TOOLS,
                    reasoning_effort="medium",
                    stream=False,
                )
                break

            except AuthenticationError:
                print("\n❌ AUTHENTICATION FAILURE (Error 401): Check your API key.")
                return ""
            except RateLimitError:
                print("\n⚠️ RATE LIMIT FAILURE (Error 429): Too many requests.")
                return ""
            except APIStatusError as e:
                if e.status_code == 402:
                    print("\n⚠️ INSUFFICIENT BALANCE (Error 402): Top up your account.")
                else:
                    print(f"\n❌ API ERROR {e.status_code}: {e}")
                return ""
            except APIConnectionError as e:
                if attempt < MAX_RETRIES - 1:
                    wait_time = 2 ** attempt
                    print(f"❌ CONNECTION FAILURE: {e}. Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"\n❌ CRITICAL FAILURE: Connection failed after {MAX_RETRIES} retries.")
                    return ""
            except Exception as e:
                print(f"\n❌ UNHANDLED ERROR: {e.__class__.__name__}: {e}")
                return ""

        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            for tool_call in message.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"\n  [Agent calls: {fn_name}({fn_args})]")

                if fn_name in TOOL_MAP:
                    try:
                        result = TOOL_MAP[fn_name](**fn_args)
                    except TypeError as e:
                        result = json.dumps({"error": f"Bad arguments to {fn_name}: {e}"})
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    preview = result[:240] + ('...' if len(result) > 240 else '')
                    print(f"  [Result: {preview}]")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })
            continue

        final_answer = message.content
        if verbose:
            print(f"\n{'='*60}")
            print(f"AOCC AGENT RESPONSE:")
            print(f"{'='*60}")
            print(final_answer)

        return final_answer

    return "Max iterations reached without final response."


# ============================================================
# 6. DEMO SCENARIOS
# ============================================================

def demo_disruption_scenario():
    print("\n" + "#"*60)
    print("# AOCC AGENT v4 - DeepSeek V4.1 Flash (reasoning_effort=medium)")
    print("# Multi-disruption conflict resolution")
    print("#"*60)

    # Scenario 1: Weather
    run_aocc_agent(
        "Flight AA300 is scheduled. Check the weather at LAX and JFK, "
        "and recommend a proactive plan."
    )

    # Scenario 2: Single flight swap (no conflict yet)
    run_aocc_agent(
        "AA100 has a hydraulic leak at the gate at JFK and needs an aircraft swap. "
        "Check available aircraft and crew (filter by replacement type), and estimate "
        "the cost if we delay the flight by 90 minutes instead of swapping."
    )

    # Scenario 3: Second AOG appears — conflict emerges
    run_aocc_agent(
        "AA200 (JFK-ORD, B787) has now also gone AOG at JFK with a pressurization fault. "
        "Check the resources at JFK and determine the impact."
    )

    # Scenario 4: THE CONFLICT — explicit allocation decision
    run_aocc_agent(
        "CRITICAL DECISION: Both AA100 (JFK-LAX) and AA200 (JFK-ORD) are AOG at JFK. "
        "The only serviceable unassigned aircraft is N999AA (A321). "
        "Resolve the resource conflict. Determine which flight gets the spare, "
        "document the losing flight's path forward, and flag any human decisions required."
    )

    # Scenario 5: Unclassified anomaly — escalation trigger
    run_aocc_agent(
        "An unclassified sensor anomaly is reported on AA100. "
        "There is no MEL reference and no maintenance disposition available. "
        "What is your recommendation?"
    )


# ============================================================
# 7. MAIN
# ============================================================

if __name__ == "__main__":
    if not DEEPSEEK_API_KEY:
        print("ERROR: DEEPSEEK_API_KEY not found in Colab userdata.")
        print("Add it via Colab sidebar → 🔑 Secrets → DEEPSEEK_API_KEY")
    else:
        demo_disruption_scenario()


############################################################
# AOCC AGENT v4 - DeepSeek V4.1 Flash (reasoning_effort=medium)
# Multi-disruption conflict resolution
############################################################

AOCC CONTROLLER: Flight AA300 is scheduled. Check the weather at LAX and JFK, and recommend a proactive plan.

  [Agent calls: get_flight_status({'flight_number': 'AA300'})]
  [Result: {"flight": "AA300", "route": "LAX-JFK", "tail": "N102AA", "type": "B737", "status": "scheduled", "departure": "2026-09-18T16:00", "pax_load": 160, "priority": "low"}]

  [Agent calls: check_weather({'airport_code': 'LAX'})]
  [Result: {"airport": "LAX", "condition": "Clear", "severity": "low", "visibility": "10 miles"}]

  [Agent calls: check_weather({'airport_code': 'JFK'})]
  [Result: {"airport": "JFK", "condition": "Thunderstorms", "severity": "high", "visibility": "2 miles"}]

  [Agent calls: get_aircraft_by_tail({'tail': 'N102AA'})]
  [Result: {"tail": "N102AA", "type": "B737"

## CASE5

In [6]:
import os
import time
import json
from openai import OpenAI
from openai import APIConnectionError, AuthenticationError, RateLimitError, APIStatusError

# --- Configuration (Reference contract from diagnostic script) ---
from google.colab import userdata
DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_MODEL = "deepseek-flash"  # DeepSeek V4.1 Flash
MAX_RETRIES = 3


# ============================================================
# 1. AOCC SIMULATED DATA SOURCES
# ============================================================
# NOTE: FLEET.serviceability = airframe condition only. Assignment is derived
# from FLIGHTS. Use find_available_aircraft() or get_tail_assignments() to
# determine true spare status. Use get_network_spares() for cross-station search.

FLEET = [
    # JFK tails
    {"tail": "N101AA", "type": "A320", "location": "JFK", "serviceability": "maintenance"},  # AA100 AOG
    {"tail": "N201AA", "type": "A350", "location": "JFK", "serviceability": "maintenance"},
    {"tail": "N202AA", "type": "B787", "location": "JFK", "serviceability": "maintenance"},  # AA200 AOG
    {"tail": "N999AA", "type": "A321", "location": "JFK", "serviceability": "serviceable"},  # ONLY JFK SPARE

    # LAX tails
    {"tail": "N102AA", "type": "B737", "location": "LAX", "serviceability": "serviceable"},  # AA300
    {"tail": "N998AA", "type": "A320", "location": "LAX", "serviceability": "serviceable"},
    {"tail": "N995AA", "type": "A321", "location": "LAX", "serviceability": "serviceable"},  # Network spare

    # ORD tails
    {"tail": "N997AA", "type": "B787", "location": "ORD", "serviceability": "serviceable"},  # Network spare

    # DFW tails
    {"tail": "N996AA", "type": "B737", "location": "DFW", "serviceability": "serviceable"},  # Network spare
]

CREW = [
    {"id": "CPT001", "role": "Captain", "base": "JFK", "type_ratings": ["A320", "A321"], "duty_hours_remaining": 8.5},
    {"id": "FO001", "role": "First Officer", "base": "JFK", "type_ratings": ["A320", "A321"], "duty_hours_remaining": 9.0},
    {"id": "FA001", "role": "Flight Attendant", "base": "JFK", "type_ratings": ["A320", "A321", "B737", "B787"], "duty_hours_remaining": 10.0},
    {"id": "CPT002", "role": "Captain", "base": "LAX", "type_ratings": ["B737"], "duty_hours_remaining": 3.0},
    {"id": "FO002", "role": "First Officer", "base": "LAX", "type_ratings": ["B737"], "duty_hours_remaining": 7.5},
    {"id": "FA002", "role": "Flight Attendant", "base": "LAX", "type_ratings": ["A320", "A321", "B737", "B787"], "duty_hours_remaining": 9.0},
    {"id": "CPT003", "role": "Captain", "base": "ORD", "type_ratings": ["B787"], "duty_hours_remaining": 9.5},  # Ferry crew
    {"id": "FO003", "role": "First Officer", "base": "ORD", "type_ratings": ["B787"], "duty_hours_remaining": 9.0},
]

FLIGHTS = [
    {"flight": "AA100", "route": "JFK-LAX", "tail": "N101AA", "type": "A320",
     "status": "AOG", "departure": "2026-09-18T14:00", "pax_load": 178, "priority": "high"},
    {"flight": "AA200", "route": "JFK-ORD", "tail": "N202AA", "type": "B787",
     "status": "AOG", "departure": "2026-09-18T15:30", "pax_load": 240, "priority": "medium"},
    {"flight": "AA300", "route": "LAX-JFK", "tail": "N102AA", "type": "B737",
     "status": "scheduled", "departure": "2026-09-18T16:00", "pax_load": 160, "priority": "low"},
]

WEATHER = {
    "JFK": {"condition": "Thunderstorms", "severity": "high", "visibility": "2 miles"},
    "LAX": {"condition": "Clear", "severity": "low", "visibility": "10 miles"},
    "ORD": {"condition": "Light Rain", "severity": "medium", "visibility": "5 miles"},
    "DFW": {"condition": "Clear", "severity": "low", "visibility": "10 miles"},
}


# ============================================================
# 2. TOOL DEFINITIONS
# ============================================================

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_flight_status",
            "description": "Get the current status of a specific flight including delay, tail, type, route, pax load, and priority.",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {"type": "string", "description": "Flight number, e.g., AA100"},
                },
                "required": ["flight_number"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_aircraft_by_tail",
            "description": "Look up aircraft details by tail number, including type, current location, and serviceability. NOTE: serviceability does not indicate whether the tail is assigned to a flight.",
            "parameters": {
                "type": "object",
                "properties": {
                    "tail": {"type": "string", "description": "Aircraft tail number, e.g., N101AA"},
                },
                "required": ["tail"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_tail_assignments",
            "description": "List all tails physically present at an airport with their assignment status (assigned flight or unassigned spare).",
            "parameters": {
                "type": "object",
                "properties": {
                    "airport_code": {"type": "string", "description": "IATA airport code, e.g., JFK"},
                },
                "required": ["airport_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_weather",
            "description": "Check current weather conditions at an airport.",
            "parameters": {
                "type": "object",
                "properties": {
                    "airport_code": {"type": "string", "description": "IATA airport code, e.g., JFK"},
                },
                "required": ["airport_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "find_available_aircraft",
            "description": "Find true spare aircraft at a specific airport (serviceable AND not assigned to any flight).",
            "parameters": {
                "type": "object",
                "properties": {
                    "airport_code": {"type": "string", "description": "Airport to search, e.g., JFK"},
                },
                "required": ["airport_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_network_spares",
            "description": "Search the entire network for serviceable, unassigned aircraft of a specific type, excluding a given airport. Use when local recovery is exhausted.",
            "parameters": {
                "type": "object",
                "properties": {
                    "aircraft_type": {"type": "string", "description": "Required aircraft type, e.g., B787"},
                    "exclude_airport": {"type": "string", "description": "Optional: airport to exclude (usually the disrupted station)"},
                },
                "required": ["aircraft_type"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "propose_ferry",
            "description": "Propose ferrying a spare aircraft from one station to another to cover a disrupted flight. Returns feasibility and estimated transit time.",
            "parameters": {
                "type": "object",
                "properties": {
                    "tail": {"type": "string", "description": "Aircraft tail to ferry"},
                    "from_airport": {"type": "string", "description": "Origin station"},
                    "to_airport": {"type": "string", "description": "Destination station"},
                    "reason": {"type": "string", "description": "Operational justification"},
                },
                "required": ["tail", "from_airport", "to_airport", "reason"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_crew_availability",
            "description": "Check crew availability, duty time remaining, and type ratings at a base airport. Optionally filter by required aircraft type.",
            "parameters": {
                "type": "object",
                "properties": {
                    "base": {"type": "string", "description": "Crew base airport, e.g., JFK"},
                    "aircraft_type": {"type": "string", "description": "Optional: filter crew by type rating, e.g., A321"},
                },
                "required": ["base"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_passenger_load",
            "description": "Get passenger count and priority for a flight. Use this when comparing flights for resource allocation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {"type": "string", "description": "Flight number, e.g., AA100"},
                },
                "required": ["flight_number"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "resolve_resource_conflict",
            "description": "When multiple flights claim the same spare resource, apply the conflict-resolution policy and return a ranked allocation. MUST be used whenever 2+ flights claim one tail.",
            "parameters": {
                "type": "object",
                "properties": {
                    "resource_tail": {"type": "string", "description": "The contested tail number"},
                    "claiming_flights": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of flight numbers claiming the resource",
                    },
                },
                "required": ["resource_tail", "claiming_flights"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "propose_recovery_options",
            "description": "Generate ranked recovery options for a disrupted flight.",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {"type": "string", "description": "Disrupted flight number"},
                    "issue": {"type": "string", "description": "Description of the disruption"},
                },
                "required": ["flight_number", "issue"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "estimate_delay_cost",
            "description": "Estimate the operational cost of a delay. If aircraft_type is omitted, the tool resolves it from the flight's tail.",
            "parameters": {
                "type": "object",
                "properties": {
                    "delay_minutes": {"type": "integer", "description": "Delay duration in minutes"},
                    "aircraft_type": {"type": "string", "description": "Optional: aircraft type, e.g., A320."},
                    "flight_number": {"type": "string", "description": "Optional: flight number for automatic type lookup."},
                },
                "required": ["delay_minutes"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "escalate_to_human",
            "description": "Formally escalate a decision to a human controller when the agent cannot safely recommend an action.",
            "parameters": {
                "type": "object",
                "properties": {
                    "reason": {"type": "string", "description": "Why human intervention is required"},
                    "flight_number": {"type": "string", "description": "Related flight, if applicable"},
                },
                "required": ["reason"],
            },
        },
    },
]


# ============================================================
# 3. TOOL IMPLEMENTATIONS
# ============================================================

def get_flight_status(flight_number: str) -> str:
    for f in FLIGHTS:
        if f["flight"] == flight_number:
            return json.dumps(f)
    return json.dumps({"error": f"Flight {flight_number} not found"})


def get_aircraft_by_tail(tail: str) -> str:
    for ac in FLEET:
        if ac["tail"].upper() == tail.upper():
            return json.dumps(ac)
    return json.dumps({"error": f"Aircraft {tail} not found"})


def get_tail_assignments(airport_code: str) -> str:
    airport = airport_code.upper()
    assigned_map = {f["tail"]: f["flight"] for f in FLIGHTS if f["status"] in ("scheduled", "delayed", "AOG")}

    result = []
    for ac in FLEET:
        if ac["location"] != airport:
            continue
        result.append({
            "tail": ac["tail"],
            "type": ac["type"],
            "serviceability": ac["serviceability"],
            "assignment": assigned_map.get(ac["tail"], None),
        })

    return json.dumps({"airport": airport, "tails": result})


def check_weather(airport_code: str) -> str:
    weather = WEATHER.get(airport_code.upper())
    if weather:
        return json.dumps({"airport": airport_code, **weather})
    return json.dumps({"error": f"Weather data for {airport_code} not available"})


def find_available_aircraft(airport_code: str) -> str:
    assigned_tails = {f["tail"] for f in FLIGHTS if f["status"] in ("scheduled", "delayed", "AOG")}
    available = [
        ac for ac in FLEET
        if ac["location"] == airport_code.upper()
        and ac["serviceability"] == "serviceable"
        and ac["tail"] not in assigned_tails
    ]
    return json.dumps({"airport": airport_code, "available_aircraft": available})


def get_network_spares(aircraft_type: str, exclude_airport: str = None) -> str:
    """Find serviceable unassigned tails of a type across the network."""
    assigned_tails = {f["tail"] for f in FLIGHTS if f["status"] in ("scheduled", "delayed", "AOG")}
    exclude = exclude_airport.upper() if exclude_airport else None

    spares = [
        ac for ac in FLEET
        if ac["type"] == aircraft_type
        and ac["serviceability"] == "serviceable"
        and ac["tail"] not in assigned_tails
        and (exclude is None or ac["location"] != exclude)
    ]

    return json.dumps({
        "searched_type": aircraft_type,
        "excluded_airport": exclude_airport,
        "network_spares": spares,
        "count": len(spares),
    })


def propose_ferry(tail: str, from_airport: str, to_airport: str, reason: str) -> str:
    """Propose a ferry movement. Returns feasibility assessment."""
    ac = next((a for a in FLEET if a["tail"].upper() == tail.upper()), None)
    if not ac:
        return json.dumps({"error": f"Aircraft {tail} not found"})

    if ac["location"].upper() != from_airport.upper():
        return json.dumps({
            "feasible": False,
            "error": f"{tail} is at {ac['location']}, not {from_airport}",
        })

    # Rough transit time estimate by route
    transit_hours = {
        ("ORD", "JFK"): 2.5, ("DFW", "JFK"): 3.5,
        ("LAX", "JFK"): 5.5, ("ORD", "LAX"): 4.5,
        ("DFW", "ORD"): 2.0, ("LAX", "ORD"): 4.0,
    }.get((from_airport.upper(), to_airport.upper()), 3.0)

    return json.dumps({
        "feasible": True,
        "tail": tail,
        "type": ac["type"],
        "from": from_airport,
        "to": to_airport,
        "estimated_transit_hours": transit_hours,
        "reason": reason,
        "note": "Ferry is non-revenue; requires crew + ATC slot. Cost driver: fuel + crew time.",
        "requires_human_approval": True,
    })


def check_crew_availability(base: str, aircraft_type: str = None) -> str:
    available = [
        c for c in CREW
        if c["base"] == base.upper()
        and c["duty_hours_remaining"] > 2.0
        and (aircraft_type is None or aircraft_type in c.get("type_ratings", []))
    ]
    return json.dumps({"base": base, "filter_type": aircraft_type, "available_crew": available})


def get_passenger_load(flight_number: str) -> str:
    flight = next((f for f in FLIGHTS if f["flight"] == flight_number), None)
    if not flight:
        return json.dumps({"error": f"Flight {flight_number} not found"})
    return json.dumps({
        "flight": flight_number,
        "pax_load": flight.get("pax_load", 0),
        "priority": flight.get("priority", "unknown"),
        "note": "Passenger load affects rebooking cost and compensation liability",
    })


def resolve_resource_conflict(resource_tail: str, claiming_flights: list) -> str:
    """
    Apply AOCC conflict-resolution policy.

    Policy:
      1. AOG status (all AOG claims tie here)
      2. Departure time (asc) — earliest departure gets priority
      3. Passenger load (desc) — tiebreaker
      4. Priority (high > medium > low) — final tiebreaker
    """
    claims = []
    for fn in claiming_flights:
        flight = next((f for f in FLIGHTS if f["flight"] == fn), None)
        if not flight:
            continue
        claims.append({
            "flight": fn,
            "status": flight["status"],
            "pax_load": flight.get("pax_load", 0),
            "priority": flight.get("priority", "unknown"),
            "departure": flight["departure"],
            "type": flight["type"],
        })

    priority_rank = {"high": 0, "medium": 1, "low": 2, "unknown": 3}

    def rank_key(c):
        aog_score = 0 if c["status"] == "AOG" else 1
        return (
            aog_score,
            c["departure"],
            -c["pax_load"],
            priority_rank.get(c["priority"], 3),
        )

    claims.sort(key=rank_key)

    winner = claims[0] if claims else None
    losers = claims[1:] if len(claims) > 1 else []

    return json.dumps({
        "resource": resource_tail,
        "winner": winner,
        "losers": losers,
        "policy": "AOG > departure_urgency (asc) > pax_load (desc) > priority",
        "rationale": "All claims AOG; allocation driven by schedule urgency first, passenger impact as tiebreaker.",
    })


def propose_recovery_options(flight_number: str, issue: str) -> str:
    flight = next((f for f in FLIGHTS if f["flight"] == flight_number), None)
    if not flight:
        return json.dumps({"error": f"Flight {flight_number} not found"})

    issue_lower = issue.lower()
    options = []

    weather_keywords = ["weather", "thunderstorm", "storm", "rain", "snow", "fog",
                        "convective", "wind", "ceiling", "visibility", "ground stop"]
    mech_keywords = ["mechanical", "aircraft", "engine", "hydraulic", "avionics",
                     "mel", "fault", "maintenance", "inspection", "aog", "pressurization"]
    crew_keywords = ["crew", "duty", "rest", "fatigue", "ftl", "captain", "first officer"]

    if any(k in issue_lower for k in weather_keywords):
        options.append({"option": "Hold for convective passage",
                        "impact": "Passenger inconvenience; recheck weather every 15-30 min",
                        "cost_driver": "Delay cost"})
        options.append({"option": "Request alternate departure slot / reroute",
                        "impact": "Low incremental cost; reduces exposure to flow programs",
                        "cost_driver": "Minimal"})
        options.append({"option": "Reroute to alternate airport",
                        "impact": "Fuel cost, passenger rebooking/ground transport",
                        "cost_driver": "Fuel + rebooking"})

    if any(k in issue_lower for k in mech_keywords):
        options.append({"option": "Repair in place",
                        "impact": "Maintenance coordination; delay duration unknown until diagnosis",
                        "cost_driver": "Delay cost + maintenance labor"})
        options.append({"option": "Swap aircraft from available fleet",
                        "impact": "Maintenance coordination; reconfiguration may be required",
                        "cost_driver": "Delay cost + reconfiguration"})
        options.append({"option": "Ferry a compatible spare from another station",
                        "impact": "Non-revenue positioning flight; requires crew + ATC slot",
                        "cost_driver": "Fuel + crew + delay until arrival"})
        options.append({"option": "Cancel and rebook passengers",
                        "impact": "High customer impact; last resort",
                        "cost_driver": "Rebooking + compensation"})

    if any(k in issue_lower for k in crew_keywords):
        options.append({"option": "Reserve crew from standby pool",
                        "impact": "Depends on standby availability and rest compliance",
                        "cost_driver": "Reserve crew premium"})
        options.append({"option": "Delay until assigned crew becomes legal",
                        "impact": "Delay duration driven by FTL rules",
                        "cost_driver": "Delay cost"})

    if not options:
        options.append({"option": "Manual review required",
                        "impact": "Issue category not auto-classified — controller assessment needed",
                        "cost_driver": "Unknown"})

    return json.dumps({"flight": flight_number, "options": options})


def estimate_delay_cost(delay_minutes: int, aircraft_type: str = None, flight_number: str = None) -> str:
    resolved_type = aircraft_type
    type_source = "provided"

    if not resolved_type and flight_number:
        flight = next((f for f in FLIGHTS if f["flight"] == flight_number), None)
        if flight:
            resolved_type = flight.get("type")
            type_source = "resolved_from_flight"

    if not resolved_type:
        resolved_type = "A320"
        type_source = "default_fallback"

    base_cost_per_min = 75
    if resolved_type in ("A350", "B787", "B777", "A330"):
        base_cost_per_min = 150

    total = delay_minutes * base_cost_per_min
    return json.dumps({
        "delay_minutes": delay_minutes,
        "aircraft_type": resolved_type,
        "type_source": type_source,
        "estimated_cost_usd": total,
        "note": "Rough estimate; excludes passenger compensation, misconnects, downstream rotation"
    })


def escalate_to_human(reason: str, flight_number: str = None) -> str:
    return json.dumps({
        "escalation": True,
        "reason": reason,
        "flight_number": flight_number,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "status": "queued_for_controller_review",
    })


TOOL_MAP = {
    "get_flight_status": get_flight_status,
    "get_aircraft_by_tail": get_aircraft_by_tail,
    "get_tail_assignments": get_tail_assignments,
    "check_weather": check_weather,
    "find_available_aircraft": find_available_aircraft,
    "get_network_spares": get_network_spares,
    "propose_ferry": propose_ferry,
    "check_crew_availability": check_crew_availability,
    "get_passenger_load": get_passenger_load,
    "resolve_resource_conflict": resolve_resource_conflict,
    "propose_recovery_options": propose_recovery_options,
    "estimate_delay_cost": estimate_delay_cost,
    "escalate_to_human": escalate_to_human,
}


# ============================================================
# 4. SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """You are an AI agent for an Airline Operations Control Center (AOCC).

Your role is to assist operations controllers in managing flight disruptions.
You have access to tools that provide real-time data about flights, weather, aircraft, and crew.

When responding to a disruption:
1. Gather all relevant data using your tools (flight status, weather, available resources)
2. Assess the severity and propagation risk
3. Propose ranked recovery options with cost estimates
4. Use escalate_to_human when a decision exceeds your authority or data is insufficient

CONFLICT RESOLUTION (critical):
- When TWO OR MORE flights claim the same spare resource, you MUST call resolve_resource_conflict.
- NEVER allocate a single tail to two flights.
- The tool returns a winner and losers based on policy: AOG > departure urgency > pax load > priority.
- Document the losing flight's path forward (delay, cancel, source alternate resource, escalate).
- If the losing flight's outcome is unacceptable (e.g., cancellation without alternatives), escalate to human.

NETWORK RECOVERY (critical):
- When local recovery is exhausted (no true spare at the disrupted station),
  you MUST call get_network_spares for the required type before accepting cancellation.
- If a network spare exists, use propose_ferry to assess transit feasibility.
- Compare: ferry cost/time vs. delay cost vs. cancellation cost.
- A ferry requires human approval — always flag it as such.
- Only recommend cancellation if NO viable ferry path exists.

Guidelines:
- Be concise and operational in your responses
- Prioritize safety and regulatory compliance
- Consider passenger impact alongside operational cost
- When a swap is proposed, verify a TRUE spare exists via find_available_aircraft
  and check type compatibility (widebody vs narrowbody downgauge requires authority sign-off)
- When checking crew for a swap, filter by the replacement aircraft's type rating
- When estimating cost, use the flight's actual aircraft type via flight_number lookup
- Do NOT run more than TWO cost scenarios per flight unless the controller asks for more
- Always recommend human verification before execution

Data model notes:
- FLEET.serviceability indicates airframe condition only (serviceable/maintenance).
  It does NOT mean the tail is unassigned. Use find_available_aircraft,
  get_tail_assignments, or get_network_spares to determine true availability.

Current operational context: JFK has a single serviceable unassigned spare (N999AA, A321).
Two flights (AA100, AA200) are AOG at JFK. Network spares may exist at other stations."""


# ============================================================
# 5. AGENT LOOP
# ============================================================

def run_aocc_agent(user_query: str, verbose: bool = True) -> str:
    if not DEEPSEEK_API_KEY:
        print("❌ CRITICAL FAILURE: API Key not loaded.")
        return ""

    client = OpenAI(
        api_key=DEEPSEEK_API_KEY,
        base_url=DEEPSEEK_BASE_URL,
        timeout=60.0,
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"AOCC CONTROLLER: {user_query}")
        print(f"{'='*60}")

    max_iterations = 15

    for iteration in range(max_iterations):
        for attempt in range(MAX_RETRIES):
            try:
                response = client.chat.completions.create(
                    model=DEEPSEEK_MODEL,
                    messages=messages,
                    tools=TOOLS,
                    reasoning_effort="medium",
                    stream=False,
                )
                break

            except AuthenticationError:
                print("\n❌ AUTHENTICATION FAILURE (Error 401): Check your API key.")
                return ""
            except RateLimitError:
                print("\n⚠️ RATE LIMIT FAILURE (Error 429): Too many requests.")
                return ""
            except APIStatusError as e:
                if e.status_code == 402:
                    print("\n⚠️ INSUFFICIENT BALANCE (Error 402): Top up your account.")
                else:
                    print(f"\n❌ API ERROR {e.status_code}: {e}")
                return ""
            except APIConnectionError as e:
                if attempt < MAX_RETRIES - 1:
                    wait_time = 2 ** attempt
                    print(f"❌ CONNECTION FAILURE: {e}. Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"\n❌ CRITICAL FAILURE: Connection failed after {MAX_RETRIES} retries.")
                    return ""
            except Exception as e:
                print(f"\n❌ UNHANDLED ERROR: {e.__class__.__name__}: {e}")
                return ""

        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            for tool_call in message.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"\n  [Agent calls: {fn_name}({fn_args})]")

                if fn_name in TOOL_MAP:
                    try:
                        result = TOOL_MAP[fn_name](**fn_args)
                    except TypeError as e:
                        result = json.dumps({"error": f"Bad arguments to {fn_name}: {e}"})
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    preview = result[:240] + ('...' if len(result) > 240 else '')
                    print(f"  [Result: {preview}]")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })
            continue

        final_answer = message.content
        if verbose:
            print(f"\n{'='*60}")
            print(f"AOCC AGENT RESPONSE:")
            print(f"{'='*60}")
            print(final_answer)

        return final_answer

    return "Max iterations reached without final response."


# ============================================================
# 6. DEMO SCENARIOS
# ============================================================

def demo_disruption_scenario():
    print("\n" + "#"*60)
    print("# AOCC AGENT v5 - DeepSeek V4.1 Flash (reasoning_effort=medium)")
    print("# Network-level recovery")
    print("#"*60)

    # Scenario 1: Proactive monitoring
    run_aocc_agent(
        "Flight AA300 is scheduled. Check the weather at LAX and JFK, "
        "and recommend a proactive plan."
    )

    # Scenario 2: Single AOG — swap feasibility
    run_aocc_agent(
        "AA100 has a hydraulic leak at the gate at JFK and needs an aircraft swap. "
        "Check available aircraft and crew (filter by replacement type), and estimate "
        "the cost if we delay the flight by 90 minutes instead of swapping."
    )

    # Scenario 3: Second AOG — conflict emerges
    run_aocc_agent(
        "AA200 (JFK-ORD, B787) has now also gone AOG at JFK with a pressurization fault. "
        "Check the resources at JFK and determine the impact."
    )

    # Scenario 4: Explicit conflict resolution
    run_aocc_agent(
        "CRITICAL DECISION: Both AA100 (JFK-LAX) and AA200 (JFK-ORD) are AOG at JFK. "
        "The only serviceable unassigned aircraft is N999AA (A321). "
        "Resolve the resource conflict. Determine which flight gets the spare, "
        "document the losing flight's path forward, and flag any human decisions required."
    )

    # Scenario 5: Unclassified anomaly
    run_aocc_agent(
        "An unclassified sensor anomaly is reported on AA100. "
        "There is no MEL reference and no maintenance disposition available. "
        "What is your recommendation?"
    )

    # Scenario 6: NETWORK RECOVERY — the v5 stress test
    run_aocc_agent(
        "AA200 (JFK-ORD, B787) is AOG at JFK and lost the N999AA conflict to AA100. "
        "There is no widebody spare at JFK. Search the network for a B787 spare, "
        "assess whether a ferry is feasible, and compare ferry cost against the "
        "delay/cancellation exposure. Recommend the best path forward."
    )


# ============================================================
# 7. MAIN
# ============================================================

if __name__ == "__main__":
    if not DEEPSEEK_API_KEY:
        print("ERROR: DEEPSEEK_API_KEY not found in Colab userdata.")
        print("Add it via Colab sidebar → 🔑 Secrets → DEEPSEEK_API_KEY")
    else:
        demo_disruption_scenario()


############################################################
# AOCC AGENT v5 - DeepSeek V4.1 Flash (reasoning_effort=medium)
# Network-level recovery
############################################################

AOCC CONTROLLER: Flight AA300 is scheduled. Check the weather at LAX and JFK, and recommend a proactive plan.

  [Agent calls: get_flight_status({'flight_number': 'AA300'})]
  [Result: {"flight": "AA300", "route": "LAX-JFK", "tail": "N102AA", "type": "B737", "status": "scheduled", "departure": "2026-09-18T16:00", "pax_load": 160, "priority": "low"}]

  [Agent calls: check_weather({'airport_code': 'LAX'})]
  [Result: {"airport": "LAX", "condition": "Clear", "severity": "low", "visibility": "10 miles"}]

  [Agent calls: check_weather({'airport_code': 'JFK'})]
  [Result: {"airport": "JFK", "condition": "Thunderstorms", "severity": "high", "visibility": "2 miles"}]

  [Agent calls: get_aircraft_by_tail({'tail': 'N102AA'})]
  [Result: {"tail": "N102AA", "type": "B737", "location": 

# AOCC Agent v5 — Summary

## Overview

A production-shape **Agentic AI decision-support system** for Airline Operations Control Center (AOCC), built on **DeepSeek V4.1 Flash** (`deepseek-flash`) with thinking mode enabled (`reasoning_effort="medium"`) and function calling.

## Architecture

| Component | Detail |
|---|---|
| **Model** | DeepSeek V4.1 Flash (`deepseek-flash`) |
| **Interface** | OpenAI-compatible Python SDK |
| **Environment** | Google Colab (`userdata.get('DEEPSEEK_API_KEY')`) |
| **Timeout** | 60 seconds |
| **Reasoning effort** | `medium` — multi-step planning |
| **Max iterations** | 15 per scenario |
| **Retry policy** | 3 attempts, exponential backoff on `APIConnectionError` |
| **Error handling** | 401 / 402 / 429 / connection / fallback |

## Tool Set (13 tools)

**Query tools:**
- `get_flight_status` — flight state, tail, type, pax, priority
- `get_aircraft_by_tail` — tail lookup with serviceability
- `get_tail_assignments` — full airport inventory with assignment status
- `get_passenger_load` — pax count and priority
- `check_weather` — airport conditions
- `check_crew_availability` — duty time, type ratings (filterable)
- `find_available_aircraft` — true local spares
- `get_network_spares` — cross-station spare search

**Planning tools:**
- `propose_recovery_options` — ranked recovery paths
- `estimate_delay_cost` — cost with auto type resolution
- `resolve_resource_conflict` — multi-claimant allocation policy
- `propose_ferry` — non-revenue repositioning feasibility

**Escalation tool:**
- `escalate_to_human` — structured queue for controller review

## Capabilities Demonstrated

| Capability | Status |
|---|---|
| **Local recovery** (swap, repair, hold) | ✅ |
| **Conflict resolution** (2 AOG flights, 1 spare) | ✅ |
| **Network recovery** (ferry from outstation) | ✅ |
| **Airworthiness gating** (no planning before MX disposition) | ✅ |
| **Crew qualification gating** (type ratings, duty limits) | ✅ |
| **Cost comparison** across recovery paths | ✅ |
| **Human-in-the-loop** at every irreversible step | ✅ |
| **Cross-scenario memory** (resource contention carried forward) | ✅ |
| **Capacity reasoning** (240 pax > A321 capacity) | ✅ |
| **Regulatory nuance** (in-family upgauge vs widebody downgauge) | ✅ |

## Run Results — 6 Scenarios

| # | Scenario | Tool Calls | Escalated | Outcome |
|---|---|---|---|---|
| 1 | AA300 proactive weather check | 9 | ✅ Crew duty blocker | Escalated |
| 2 | AA100 hydraulic leak + swap | 12 | ❌ | Swap recommended |
| 3 | AA200 pressurization AOG | 15 | ✅ No local recovery | Escalated |
| 4 | Explicit conflict resolution | 14 | ✅ Loser path documented | AA100 wins |
| 5 | Unclassified sensor anomaly | 3 | ✅ Airworthiness | Refused to plan |
| 6 | Network recovery for AA200 | 9 | ✅ Ferry approval | Ferry recommended |

**Total: 62 tool calls, 100% success rate, zero API errors.**

## Key Reasoning Highlights

**Scenario 4 — Conflict resolution:**
> *"Both are AOG, so the tiebreak fell to schedule urgency → AA100 (14:00) wins."*

Applied policy transparently and documented the losing flight's blocked path forward.

**Scenario 5 — Airworthiness gate:**
> *"I have not proposed a tail swap, ferry, or cancellation path, because the root question is whether N101AA is even dispatchable — not how to route around it."*

Refused to plan recovery before safety determination.

**Scenario 6 — Network recovery:**
> *"Ferry delay (~$31.5K) is the lowest-cost recovery and preserves the B787 gauge. Cancellation is not justified."*

Quantified three paths (ferry $31.5K, wait $54K, cancel = rebooking 240 pax) and recommended the optimal one.

**Crew gap detection (Scenarios 3–6):**
> *"JFK has NO B787-rated pilots (only FA001), so B787 flight crew must reposition/ferry in with N997AA — a crew-positioning blocker requiring human decision."*

Recognized that a ferry solves the aircraft problem but not the crew problem.

## Design Principles Enforced

1. **Safety before logistics** — no recovery planning before airworthiness is determined
2. **Policy-driven allocation** — `AOG > departure urgency > pax load > priority`
3. **No double-allocation** — `resolve_resource_conflict` mandatory when 2+ flights claim one tail
4. **Network before cancellation** — `get_network_spares` mandatory before accepting a cancel
5. **Human approval for irreversible actions** — swaps, ferries, cancellations all escalate
6. **Cost discipline** — max 2 cost scenarios per flight
7. **Type-rating awareness** — crew filtered by replacement aircraft type

## Reference Contract (from diagnostic script)

All configuration, error handling, and client setup follows the original diagnostic script exactly:
- `userdata.get('DEEPSEEK_API_KEY')`
- `base_url="https://api.deepseek.com"`
- `timeout=60.0`
- 401 / 402 / 429 / `APIConnectionError` handling
- Exponential backoff retry (`2 ** attempt`)

## Verdict

**v5 is the reference implementation** for a bounded AOCC decision-support agent. It handles the complete disruption recovery ladder — local, conflict, and network — with transparent reasoning, correct regulatory posture, and disciplined escalation. The remaining gaps are depth-of-data, not architecture.